# Data prepration

In [ ]:
!pip install -q datasets statsmodels geopandas

In [ ]:
# File Helper
from pathlib import Path
from typing import List, Union

def find_full_path(pattern: str, start_dir: Union[str, Path] = '.') -> List[Path]:
    """
    Searches recursively for files or directories matching a pattern
    starting from a given directory and returns their full absolute paths.

    Args:
        pattern: The glob pattern to search for (e.g., 'my_file.csv', '*.txt', 'my_directory').
                 Can be a specific filename, a pattern with wildcards, or a directory name.
        start_dir: The directory path (as a string or Path object) to start the search from.
                   Defaults to the current working directory ('.').

    Returns:
        A list of Path objects representing the full absolute paths of the found items.
        Returns an empty list if no items match the pattern.

    Raises:
        FileNotFoundError: If the start_dir does not exist or is not a directory.
        PermissionError: If there are permission issues accessing directories.
    """
    try:
        # Convert start_dir to a Path object and resolve to an absolute path
        base_path = Path(start_dir).resolve(strict=True) # strict=True ensures the path exists

        if not base_path.is_dir():
            raise FileNotFoundError(f"Error: The starting directory '{start_dir}' is not a valid directory.")

        # Use rglob to search recursively for the pattern
        # rglob finds files and directories matching the pattern in the directory tree
        found_paths = list(base_path.rglob(pattern))

        # Ensure all paths returned are absolute (rglob should already do this when starting from an absolute path)
        # This is more for clarity and robustness
        absolute_paths = [p.resolve() for p in found_paths]

        return absolute_paths

    except FileNotFoundError as e:
        print(f"Error finding start directory: {e}")
        return [] # Return empty list on error finding start_dir
    except PermissionError as e:
        print(f"Permission error during search: {e}")
        return [] # Return empty list on permission error
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        return [] # Return empty list on other errors

In [ ]:
from datasets import load_dataset, load_from_disk

Dataset_path = 'us_power_outages_dataset'

Dataset_name = 'mhdaw/US-PowerOutages-daily-mean'

try:
  dataset_path = find_full_path(Dataset_path)
  dataset_path = dataset_path[0]
  dataset = load_from_disk(dataset_path)
except Exception as e:
  print(f"Error loading dataset: {e}")
  print(f"Trying to load dataset from {Dataset_name} on huggingface.")
  dataset = load_dataset(Dataset_name)

In [ ]:
# convert to pandas for easier processing
df_train = ds["train"].to_pandas()
df_val = ds["validation"].to_pandas()
df_test = ds["test"].to_pandas()

## Data Processing:

Well we cannot share the full data prepration and processing code in this inference notebook as it took several notebook itself and it was memory intensive, But we can share the overall approach and the notebook links to look into:

So first we took the original datasets provided by the host(eaglei and storm events) and we merged them together in multiple time interval including: 15 minuts, hourly, daile and weekly, we used 2 method of aggregation for each of them except the 15 min, we used sum method and mean method for better understanding and study.
How ever due to computation limitation we stick to the daily and mean aggregated for this notebook.

Note that the 15 min version had more than 180 million rows!

After they were ready we needed more data sources, We tried ERA5 but that was not possible due to it size(1.7 PB!) and that it had 2 way for acceses:
1- Climate data store source, This would take more time to download each file

2- Google cloud, it is publicly available but it has a licesnse problem for us so we don't used that( And it was larger than we with normal compute could handle)


then we tried other weather sources including GHCN Daily, This data was good but it has lots of missing from both location and weather information so we discarded this as well.

We used weather data from metostat, this was the most promisig, in the repo you will see how we downloaed it, the processing was simple:
time and location filtering.

We used the shap files from all us years to process the location information(lungitude and latitue) to create the county, state and fips code.

The we had to merege these together, but as the weather data had coverd a small part of our fips code, we had to come up with a imputation sterategy, using statistical meathods doesnt made sense as the locations are every where,

So we used the locations that we had and the state and avraged the info for the missing ones, this was the most meaningfull and okay.


then we merged them based on the fips code which was RAM intensive.

At the end we added some temporal features and pushed the processed data to the hub for easier access(will take down if you want).



We also tried to add population data, infestructure data , etc but they were not easy to find and did not cover or helped with the goal so we didn't used them for final data

## Notebooks:
https://github.com/Mhdaw/NeuralStorm/tree/main/NeuralStorm/Data/Downloads

https://github.com/Mhdaw/NeuralStorm/tree/main/NeuralStorm/Data/PreProcess

# EDA

In [ ]:
#%load_ext cudf.pandas
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import ccf
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import adfuller
from scipy.stats import spearmanr, pearsonr

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.ensemble import IsolationForest
from datasets import load_dataset

import geopandas as gpd
GEOPANDAS_AVAILABLE = True

from tqdm.notebook import tqdm
import gc
import os
import warnings


# Ignore all warnings
warnings.filterwarnings("ignore")

# Set plot style and defaults
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
pd.options.display.max_columns = None # To show all

In [ ]:
df = df_test.copy() # smallest is val, we use sample to handle largness.

if 'log_customers_out' not in df.columns and 'customers_out' in df.columns:
     df['log_customers_out'] = np.log1p(df['customers_out'])

# column groups
target_col = 'customers_out'
log_target_col = 'log_customers_out'
severity_col = 'severity' # Categorical severity
weather_num_cols = ['tavg', 'tmin', 'tmax', 'prcp', 'snow', 'wspd']
event_cols = [col for col in df.columns if col.startswith('event_count')]
geo_cols = ['state', 'county', 'fips_code']
time_features = ['year', 'month', 'day', 'hour', 'dayofweek','dayofyear', 'weekofyear', 'quarter', 'season', 'is_weekend']

# Filter out non-existent columns from lists
weather_num_cols = [col for col in weather_num_cols if col in df.columns]
event_cols = [col for col in event_cols if col in df.columns]
geo_cols = [col for col in geo_cols if col in df.columns]

# Ensure severity exists and is categorical
if severity_col in df.columns and not pd.api.types.is_categorical_dtype(df[severity_col]):
    df[severity_col] = df[severity_col].astype('category')

print(f"Using Target: {target_col}, Log Target: {log_target_col}")
print(f"Weather Cols: {weather_num_cols}")
print(f"Event Cols: {len(event_cols)}")
print(f"Geo Cols: {geo_cols}")

In [ ]:
# 2. Initial Overview
print("--- Initial Overview ---")
print(f"Dataset Shape: {df.shape}")

print("\n-Data Type")
print(df.info()) # .info() is more comprehensive than .dtypes

# Ensure 'time' is datetime
if not pd.api.types.is_datetime64_any_dtype(df['time']):
    print("\nConverting 'time' column to datetime...")
    df['time'] = pd.to_datetime(df['time'], errors='coerce')
    # Check if conversion introduced NaNs
    if df['time'].isnull().any():
        print("Warning: Errors encountered during 'time' conversion, NaNs introduced.")

# Ensure 'fips_code' is treated as a string/object identifier
if pd.api.types.is_numeric_dtype(df['fips_code']):
    print("\nConverting 'fips_code' to string type...")
    # Handle potential leading zeros if read as integer
    try:
        df['fips_code'] = df['fips_code'].astype(int).astype(str).str.zfill(5)
    except:
        print("Could not convert FIPS code using zfill, converting directly to string.")
        df['fips_code'] = df['fips_code'].astype(str)
elif not pd.api.types.is_string_dtype(df['fips_code']):
     df['fips_code'] = df['fips_code'].astype(str)

print(f"\nFIPS Code data type: {df['fips_code'].dtype}")
print(f"Sample FIPS codes: {df['fips_code'].unique()[:5]}")

In [ ]:
# 3. Missing Value Analysis
print("\n--- Missing Value Analysis ---")
missing_percentage = (df.isnull().sum() / len(df)) * 100
missing_percentage = missing_percentage[missing_percentage > 0].sort_values(ascending=False)

if not missing_percentage.empty:
    print("Percentage of Missing Values per Column (> 0%):")
    print(missing_percentage)
else:
    print("No missing values found.")

In [ ]:
# 4. Duplicate Check
print("\nDuplicate Check")
n_duplicates = df.duplicated().sum()
print(f"Number of fully duplicate rows: {n_duplicates}")
if n_duplicates > 0:
    print("Investigating duplicate rows...")
    print(df[df.duplicated(keep=False)].sort_values(by=list(df.columns)).head())

In [ ]:
# 5. Basic Statistics
print("\nDescriptive Statistics (Numerical) ---")
# Identify numerical columns (excluding identifiers like year, month etc.)
numerical_cols = df.select_dtypes(include=np.number).columns.tolist()
time_features = ['year', 'month', 'day', 'hour', 'dayofweek', 'dayofyear', 'weekofyear', 'quarter']

numerical_cols_for_desc = [col for col in numerical_cols if col not in time_features + ['is_weekend']]
print(df[numerical_cols_for_desc].describe().T)

print("\nDescriptive Statistics (Categorical/Object) ---")
categorical_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
print(df[categorical_cols].describe().T)

print("\nUnique Value Counts (Categorical/Identifiers) ---")
identifier_cols = ['fips_code', 'county', 'state', 'severity', 'season', 'weather_source']
for col in identifier_cols:
    if col in df.columns:
        print(f"\nUnique values in '{col}' (Top 5):")
        print(df[col].value_counts().head())
        print(f"Total unique values: {df[col].nunique()}")
        if df[col].nunique() > 1000 and col == 'county': # Avoid printing too many counties
             print("(Skipping full list for county)")

---
## B. Univariate Analysis (Understanding Individual Variables)
---

In [ ]:
# 1. Target Variables (`customers_out`, `severity`)
print("Univariate Analysis: Target Variables")
target_col = 'customers_out'
severity_col = 'severity'

# Distribution of customers_out
plt.figure(figsize=(14, 5))
plt.subplot(1, 2, 1)
sns.histplot(df[target_col], kde=True, bins=50)
plt.title('Distribution of Customers Out')
plt.xlabel('Customers Out')
plt.ylabel('Frequency')

# Distribution of log(customers_out + 1) - useful for highly skewed data
plt.subplot(1, 2, 2)
sns.histplot(np.log1p(df[target_col]), kde=True, bins=50)
plt.title('Distribution of log(Customers Out + 1)')
plt.xlabel('log(Customers Out + 1)')
plt.ylabel('Frequency')
plt.tight_layout()
plt.show()

# Distribution of non-zero customers_out
plt.figure(figsize=(14, 5))
plt.subplot(1, 2, 1)
non_zero_outages = df[df[target_col] > 0][target_col]
if not non_zero_outages.empty:
    sns.histplot(non_zero_outages, kde=True, bins=50)
    plt.title('Distribution of Non-Zero Customers Out')
    plt.xlabel('Customers Out (where > 0)')
    plt.ylabel('Frequency')
else:
    plt.text(0.5, 0.5, 'No non-zero outages found', ha='center', va='center')
    plt.title('Distribution of Non-Zero Customers Out')


# Distribution of log(non-zero customers_out)
plt.subplot(1, 2, 2)
if not non_zero_outages.empty:
    sns.histplot(np.log1p(non_zero_outages), kde=True, bins=50)
    plt.title('Distribution of log(Non-Zero Customers Out + 1)')
    plt.xlabel('log(Customers Out + 1) (where > 0)')
    plt.ylabel('Frequency')
else:
    plt.text(0.5, 0.5, 'No non-zero outages found', ha='center', va='center')
    plt.title('Distribution of log(Non-Zero Customers Out + 1)')

plt.tight_layout()
plt.show()


# Frequency of severity levels
if severity_col in df.columns:
    plt.figure(figsize=(10, 5))
    sns.countplot(data=df, x=severity_col, order=df[severity_col].value_counts().index)
    plt.title('Frequency of Outage Severity Levels')
    plt.xlabel('Severity')
    plt.ylabel('Count')
    plt.xticks(rotation=45)
    plt.show()
else:
    print(f"'{severity_col}' column not found.")

In [ ]:
# 2. Weather Variables (Numerical)
print("Univariate Analysis: Numerical Weather Variables")
weather_num_cols = ['tavg', 'tmin', 'tmax', 'prcp', 'snow', 'wspd']

# Plot distributions (Histograms and KDE)
print("Plotting distributions for numerical weather variables...")
n_cols = 3
n_rows = (len(weather_num_cols) + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 5, n_rows * 4))
axes = axes.flatten() # Flatten to 1D array for easy iteration

for i, col in enumerate(weather_num_cols):
    if col in df.columns:
        sns.histplot(df[col].dropna(), kde=True, ax=axes[i], bins=30)
        axes[i].set_title(f'Distribution of {col}')
        axes[i].set_xlabel(col)
        axes[i].set_ylabel('Frequency')
    else:
        axes[i].set_title(f'{col} not found')
        axes[i].axis('off')

# Hide any unused subplots
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

# Plot box plots (Identify outliers)
print("\nPlotting box plots for numerical weather variables...")
fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 5, n_rows * 4))
axes = axes.flatten()

for i, col in enumerate(weather_num_cols):
     if col in df.columns:
        sns.boxplot(y=df[col].dropna(), ax=axes[i])
        axes[i].set_title(f'Box Plot of {col}')
        axes[i].set_ylabel(col)
     else:
        axes[i].set_title(f'{col} not found')
        axes[i].axis('off')

# Hide any unused subplots
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

In [ ]:
# 3. Event Count Variables (`event_count_*`)
print("Univariate Analysis: Event Count Variables")
event_cols = [col for col in df.columns if col.startswith('event_count')]

if not event_cols:
    print("No columns starting with 'event_count' found.")
else:
    print(f"Found {len(event_cols)} event count columns.")

    # Calculate sparsity (percentage of non-zero entries)
    sparsity = {}
    for col in event_cols:
        sparsity[col] = (df[col] > 0).mean() * 100

    sparsity_series = pd.Series(sparsity).sort_values()
    print("\nSparsity (Percentage of Days with Event > 0):")
    print(sparsity_series)

    print("\nMost Frequent Event Types (Top 10 by non-zero days):")
    print(sparsity_series.tail(10))

    print("\nLeast Frequent Event Types (Bottom 10 by non-zero days):")
    print(sparsity_series.head(10))

    cols_to_plot = sparsity_series.tail(5).index.tolist() # Plot top 5 most frequent
    print(f"\nPlotting distributions for selected event types: {cols_to_plot}")

    if cols_to_plot:
        n_plots = len(cols_to_plot)
        n_cols_hist = 3
        n_rows_hist = (n_plots + n_cols_hist - 1) // n_cols_hist
        fig_hist, axes_hist = plt.subplots(n_rows_hist, n_cols_hist, figsize=(n_cols_hist * 5, n_rows_hist * 4), squeeze=False)
        axes_hist = axes_hist.flatten()

        for i, col in enumerate(cols_to_plot):
            # Use value_counts for sparse, discrete data like counts
            counts = df[col].value_counts().sort_index()
            # Limit the number of bars if there are many unique counts (e.g., > 50)
            if len(counts) > 50:
                 # For very non-sparse counts, a histplot might be better
                 sns.histplot(df[col][df[col]>0], ax=axes_hist[i], discrete=True)
                 axes_hist[i].set_title(f'Distribution of {col} (Non-Zero Counts)')

            else:
                sns.barplot(x=counts.index, y=counts.values, ax=axes_hist[i], color='skyblue')
                axes_hist[i].set_title(f'Distribution of {col}')

            axes_hist[i].set_xlabel('Event Count')
            axes_hist[i].set_ylabel('Frequency')

        # Hide any unused subplots
        for j in range(i + 1, len(axes_hist)):
            fig_hist.delaxes(axes_hist[j])

        plt.tight_layout()
        plt.show()
    else:
        print("No event columns selected for plotting.")

In [ ]:
# 4. Temporal Features
print("Univariate Analysis: Temporal Features")
temporal_features = ['year', 'month', 'dayofweek', 'season', 'is_weekend']

n_cols_temp = 3
n_rows_temp = (len(temporal_features) + n_cols_temp - 1) // n_cols_temp
fig_temp, axes_temp = plt.subplots(n_rows_temp, n_cols_temp, figsize=(n_cols_temp * 5, n_rows_temp * 4))
axes_temp = axes_temp.flatten()

for i, col in enumerate(temporal_features):
    if col in df.columns:
        order = None
        if col == 'month': order = sorted(df[col].unique())
        if col == 'dayofweek': order = sorted(df[col].unique()) # 0=Mon, 6=Sun
        if col == 'season': order = ['Winter', 'Spring', 'Summer', 'Autumn']

        sns.countplot(data=df, x=col, ax=axes_temp[i], order=order, palette='viridis')
        axes_temp[i].set_title(f'Record Count by {col}')
        axes_temp[i].set_xlabel(col)
        axes_temp[i].set_ylabel('Number of Records')
        if col in ['month', 'dayofweek', 'season']:
             axes_temp[i].tick_params(axis='x', rotation=45)
    else:
        axes_temp[i].set_title(f'{col} not found')
        axes_temp[i].axis('off')

# Hide unused subplots
for j in range(i + 1, len(axes_temp)):
    fig_temp.delaxes(axes_temp[j])

plt.tight_layout()
plt.show()

In [ ]:
# 5. Geospatial Features (`state`, `county`)
print("Univariate Analysis: Geospatial Features")

# Records per State
if 'state' in df.columns:
    plt.figure(figsize=(15, 7))
    state_counts = df['state'].value_counts()
    sns.barplot(x=state_counts.index, y=state_counts.values, palette='coolwarm')
    plt.title('Number of Records (County-Days) per State')
    plt.xlabel('State')
    plt.ylabel('Record Count')
    plt.xticks(rotation=90)
    plt.show()
else:
    print("'state' column not found.")

# Records per County (Top N)
if 'county' in df.columns:
    county_counts = df['county'].value_counts()
    n_top_counties = 30
    print(f"\nTop {n_top_counties} Counties by Record Count:")
    print(county_counts.head(n_top_counties))

    if county_counts.nunique() > 1: # Check if there's more than one county to plot
        plt.figure(figsize=(15, 7))
        top_counties = county_counts.head(n_top_counties)
        sns.barplot(x=top_counties.index, y=top_counties.values, palette='coolwarm')
        plt.title(f'Top {n_top_counties} Counties by Record Count')
        plt.xlabel('County')
        plt.ylabel('Record Count')
        plt.xticks(rotation=90)
        plt.show()
    else:
        print("Only one unique county found, skipping county plot.")
else:
    print("'county' column not found.")

---
## C. Bivariate Analysis (Relationships between Two Variables)
---
Using log-transformed target for visualization where appropriate due to skewness

In [ ]:
df['log_customers_out'] = np.log1p(df[target_col])

# 1. Target vs. Numerical Weather
print("Bivariate Analysis: Target vs. Numerical Weather")

# Scatter plots
sample_fraction = 0.1
df_sample = df.sample(frac=sample_fraction, random_state=42) if len(df) > 50000 else df

print(f"Plotting scatter plots for Target vs Weather (using {sample_fraction*100:.1f}% sample if large)...")
n_cols_scatter = 3
n_rows_scatter = (len(weather_num_cols) + n_cols_scatter - 1) // n_cols_scatter
fig_scatter, axes_scatter = plt.subplots(n_rows_scatter, n_cols_scatter, figsize=(n_cols_scatter * 5, n_rows_scatter * 4))
axes_scatter = axes_scatter.flatten()

for i, col in enumerate(weather_num_cols):
    if col in df.columns:
        sns.scatterplot(data=df_sample, x=col, y='log_customers_out', ax=axes_scatter[i], alpha=0.3, s=10)
        axes_scatter[i].set_title(f'log({target_col}+1) vs {col}')
        axes_scatter[i].set_xlabel(col)
        axes_scatter[i].set_ylabel(f'log({target_col}+1)')
    else:
        axes_scatter[i].set_title(f'{col} not found')
        axes_scatter[i].axis('off')

# Hide unused subplots
for j in range(i + 1, len(axes_scatter)):
    fig_scatter.delaxes(axes_scatter[j])

plt.tight_layout()
plt.show()

# Correlation Matrix (Numerical Weather + Target)
print("\nCalculating Correlation Matrix (Spearman for robustness to outliers/non-linearity)...")
cols_for_corr = weather_num_cols + [target_col, 'log_customers_out']
valid_cols_for_corr = [col for col in cols_for_corr if col in df.columns and pd.api.types.is_numeric_dtype(df[col])]

if len(valid_cols_for_corr) > 1:
    correlation_matrix = df[valid_cols_for_corr].corr(method='spearman') # Spearman is often better for skewed data or non-linear relationships

    plt.figure(figsize=(10, 8))
    sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=.5)
    plt.title('Spearman Correlation Matrix (Weather Variables and Target)')
    plt.show()

    print("\nCorrelations with target variable:")
    print(correlation_matrix[target_col].sort_values(ascending=False))
    print("\nCorrelations with log-transformed target variable:")
    print(correlation_matrix['log_customers_out'].sort_values(ascending=False))
else:
    print("Not enough valid numerical columns found for correlation matrix.")

In [ ]:
# 2. Target vs. Event Counts
print("Bivariate Analysis: Target vs. Event Counts")

if event_cols:
    # Correlation Analysis (Target vs. Event Counts)
    print("\nCalculating Correlations (Spearman) between Target and Event Counts...")
    event_corr_cols = event_cols + [target_col, 'log_customers_out']
    valid_event_corr_cols = [col for col in event_corr_cols if col in df.columns and pd.api.types.is_numeric_dtype(df[col])]

    if len(valid_event_corr_cols) > 1:
        event_correlation_matrix = df[valid_event_corr_cols].corr(method='spearman')
        target_event_corr = event_correlation_matrix[[target_col, 'log_customers_out']].drop([target_col, 'log_customers_out'])
        print(f"\nTop 10 Event Counts Correlated with {target_col}:")
        print(target_event_corr[target_col].abs().sort_values(ascending=False).head(10))
        print(f"\nTop 10 Event Counts Correlated with log({target_col}+1):")
        print(target_event_corr['log_customers_out'].abs().sort_values(ascending=False).head(10))

        top_n_events = target_event_corr['log_customers_out'].abs().sort_values(ascending=False).head(15).index.tolist()
        if top_n_events:
            plt.figure(figsize=(8, 10))
            sns.heatmap(target_event_corr.loc[top_n_events], annot=True, cmap='coolwarm', fmt=".2f", linewidths=.5)
            plt.title(f'Spearman Correlation: Target vs Top {len(top_n_events)} Event Counts')
            plt.show()

    else:
        print("Not enough valid event count columns found for correlation analysis.")


    events_to_analyze = target_event_corr['log_customers_out'].abs().sort_values(ascending=False).head(5).index.tolist()
    print(f"\nComparing Outages for Days With vs Without Specific Events: {events_to_analyze}")

    for event in events_to_analyze:
        if event in df.columns:
            plt.figure(figsize=(10, 5))
            # Using log scale for y-axis due to skewness
            sns.boxplot(x=(df[event] > 0), y=df['log_customers_out'])
            plt.title(f'log({target_col}+1) Distribution: {event} Occurred vs Not Occurred')
            plt.xlabel(f'{event} > 0')
            plt.ylabel(f'log({target_col}+1)')
            # Add mean comparison text
            mean_with = df.loc[df[event] > 0, 'log_customers_out'].mean()
            mean_without = df.loc[df[event] == 0, 'log_customers_out'].mean()
            plt.text(0.1, 0.9, f"Mean (With): {mean_with:.2f}\nMean (Without): {mean_without:.2f}", transform=plt.gca().transAxes)
            plt.show()
        else:
            print(f"Column {event} not found for grouped analysis.")

else:
    print("No event count columns found for bivariate analysis.")

In [ ]:
# 3. Target vs. Temporal Features
print("Bivariate Analysis: Target vs. Temporal Features")

# Box plots/Violin plots: log(customers_out) grouped by temporal features
for col in temporal_features:
    if col in df.columns:
        plt.figure(figsize=(12, 6))
        order = None
        if col == 'month': order = sorted(df[col].unique())
        if col == 'dayofweek': order = sorted(df[col].unique()) # 0=Mon, 6=Sun
        if col == 'season': order = ['Winter', 'Spring', 'Summer', 'Autumn']

        # Using boxplot, violinplot is also an option
        sns.boxplot(data=df, x=col, y='log_customers_out', order=order, palette='viridis')
        plt.title(f'log({target_col}+1) by {col}')
        plt.xlabel(col)
        plt.ylabel(f'log({target_col}+1)')
        if col in ['month', 'dayofweek', 'season']:
             plt.xticks(rotation=45)
        plt.show()
    else:
        print(f"Column {col} not found for temporal analysis.")

# Line plots: Average log(customers_out) per month/season over the years
print("\nPlotting average outages over time...")
plt.figure(figsize=(14, 6))
df.groupby('month')['log_customers_out'].mean().plot(marker='o')
plt.title(f'Average log({target_col}+1) by Month (All Years)')
plt.xlabel('Month')
plt.ylabel(f'Average log({target_col}+1)')
plt.xticks(ticks=range(1, 13))
plt.grid(True)
plt.show()

plt.figure(figsize=(14, 6))
df.groupby('year')['log_customers_out'].mean().plot(marker='o')
plt.title(f'Average log({target_col}+1) by Year')
plt.xlabel('Year')
plt.ylabel(f'Average log({target_col}+1)')
plt.grid(True)
plt.show()

# Monthly pattern faceted by year
if 'year' in df.columns and 'month' in df.columns:
    monthly_avg = df.groupby(['year', 'month'])['log_customers_out'].mean().reset_index()
    g = sns.FacetGrid(monthly_avg, col='year', col_wrap=4, height=3)
    g.map(plt.plot, 'month', 'log_customers_out', marker='o')
    g.set_titles("Year: {col_name}")
    g.set_axis_labels("Month", f"Avg log({target_col}+1)")
    plt.xticks(ticks=range(1, 13, 2)) # Adjust x-ticks for clarity
    plt.suptitle(f'Average log({target_col}+1) by Month for Each Year', y=1.02)
    plt.tight_layout()
    plt.show()

In [ ]:
# 4. Target vs. Geospatial Features
print("Bivariate Analysis: Target vs. Geospatial Features")

# Aggregate customers_out by state
if 'state' in df.columns:
    state_agg = df.groupby('state').agg(
        total_outages=(target_col, 'sum'),
        mean_outages=(target_col, 'mean'),
        median_outages=(target_col, 'median'),
        max_outages=(target_col, 'max'),
        days_with_outages=(target_col, lambda x: (x > 0).sum()),
        record_count=(target_col, 'count')
    ).reset_index()
    state_agg['outage_frequency'] = state_agg['days_with_outages'] / state_agg['record_count']
    state_agg['mean_outages_when_occur'] = state_agg['total_outages'] / state_agg['days_with_outages'].replace(0, np.nan)


    print("\nTop 10 States by Total Customers Out:")
    print(state_agg.sort_values('total_outages', ascending=False).head(10))

    print("\nTop 10 States by Mean Customers Out per County-Day:")
    print(state_agg.sort_values('mean_outages', ascending=False).head(10))

    print("\nTop 10 States by Outage Frequency (Days with Outages / Total Days):")
    print(state_agg.sort_values('outage_frequency', ascending=False).head(10))

    # Plotting top N states by mean outages
    top_n_states_plot = 20
    state_agg_sorted = state_agg.sort_values('mean_outages', ascending=False).head(top_n_states_plot)
    plt.figure(figsize=(15, 7))
    sns.barplot(data=state_agg_sorted, x='state', y='mean_outages', palette='coolwarm')
    plt.title(f'Top {top_n_states_plot} States by Mean Customers Out per County-Day')
    plt.xlabel('State')
    plt.ylabel('Mean Customers Out')
    plt.xticks(rotation=90)
    plt.show()

else:
    print("'state' column not found.")

# Aggregate by County (Show top N - mapping would be better)
if 'county' in df.columns and 'state' in df.columns:
    county_agg = df.groupby(['state', 'county']).agg(
        total_outages=(target_col, 'sum'),
        mean_outages=(target_col, 'mean'),
        median_outages=(target_col, 'median'),
        max_outages=(target_col, 'max'),
         days_with_outages=(target_col, lambda x: (x > 0).sum()),
         record_count=(target_col, 'count')
    ).reset_index()
    county_agg['outage_frequency'] = county_agg['days_with_outages'] / county_agg['record_count']
    county_agg['mean_outages_when_occur'] = county_agg['total_outages'] / county_agg['days_with_outages'].replace(0, np.nan)


    print("\nTop 20 Counties by Total Customers Out:")
    print(county_agg.sort_values('total_outages', ascending=False).head(20))

    print("\nTop 20 Counties by Mean Customers Out per County-Day:")
    print(county_agg.sort_values('mean_outages', ascending=False).head(20))
    # TO DO USE GEOPANDAS!
else:
    print("'county' or 'state' column not found for county-level aggregation.")

In [ ]:
# 5. Weather vs. Weather (Multicollinearity Check)
print("Bivariate Analysis: Weather vs. Weather")

# Correlation matrix for numerical weather variables
if len(weather_num_cols) > 1:
    weather_corr_matrix = df[weather_num_cols].corr(method='pearson') # Pearson is common for weather inter-correlation

    plt.figure(figsize=(10, 8))
    sns.heatmap(weather_corr_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=.5)
    plt.title('Pearson Correlation Matrix (Numerical Weather Variables)')
    plt.show()

    print("Note: High correlations observed (e.g., between tavg, tmin, tmax) indicate multicollinearity, which should be considered during modeling.")
else:
    print("Not enough numerical weather columns for correlation matrix.")

# Scatter plots for selected pairs (wind vs precipitation)
if 'wspd' in df.columns and 'prcp' in df.columns:
    plt.figure(figsize=(8, 6))
    sns.scatterplot(data=df_sample, x='wspd', y='prcp', alpha=0.3, s=10)
    plt.title('Wind Speed vs Precipitation (Sampled)')
    plt.xlabel('Wind Speed (wspd)')
    plt.ylabel('Precipitation (prcp)')
    plt.show()

---
## D. Time Series Analysis
---

In [ ]:
print("D. Time Series Analysis")

df_ts = df.set_index('time')
if not isinstance(df_ts.index, pd.DatetimeIndex):
    print("Skipping Time Series plots: DataFrame does not have a DatetimeIndex.")
else:
    # 1. Aggregate Outages
    print("\nAggregating outages across all counties...")
    daily_total_outages = df_ts.groupby(df_ts.index)[target_col].sum()
    daily_mean_log_outages = df_ts.groupby(df_ts.index)[log_target_col].mean() # Using mean log

    # Plot Aggregated Outages
    fig, ax = plt.subplots(2, 1, figsize=(15, 10), sharex=True)
    daily_total_outages.plot(ax=ax[0], label='Total Outages')
    ax[0].set_title('Total Daily Customers Out (Sum Across All Counties)')
    ax[0].set_ylabel('Total Customers Out')
    ax[0].grid(True)

    daily_mean_log_outages.plot(ax=ax[1], label='Mean Log Outages', color='orange')
    ax[1].set_title(f'Average Daily {log_target_col} (Mean Across Reporting Counties)')
    ax[1].set_ylabel(f'Mean {log_target_col}')
    ax[1].grid(True)
    plt.xlabel('Time')
    plt.tight_layout()
    plt.show()

    daily_mean_log_outages_filled = daily_mean_log_outages.fillna(0) # Simple fill for demo
    if len(daily_mean_log_outages_filled) > 365 * 2:
        print("\nDecomposing aggregated daily mean log outage time series...")
        try:
            decomposition_outages = seasonal_decompose(daily_mean_log_outages_filled, model='additive', period=365)
            fig_decomp_outages = decomposition_outages.plot()
            fig_decomp_outages.set_size_inches(14, 10)
            plt.suptitle(f'Decomposition: Aggregated Daily Mean {log_target_col}', y=1.01)
            plt.show()
        except Exception as e:
            print(f"Could not decompose outage series: {e}")
    else:
         print("\nSkipping outage time series decomposition: Not enough data (< 2 years).")


    # --- NEW: Aggregate and Plot Weather Time Series ---
    print("\nAggregating and plotting weather time series...")
    weather_agg_ops = {
        'tavg': 'mean',
        'prcp': 'mean', # Mean precip per county reporting
        'snow': 'mean', # Mean snow per county reporting
        'wspd': ['mean', 'max'] # Avg and Max wind speed across counties
    }
    # Filter ops for columns actually present in the DataFrame
    valid_weather_agg_ops = {k: v for k, v in weather_agg_ops.items() if k in df.columns}

    if valid_weather_agg_ops:
        daily_agg_weather = df_ts.groupby(df_ts.index)[weather_num_cols].agg(valid_weather_agg_ops)
        # Flatten MultiIndex columns (e.g., ('wspd', 'mean') -> 'wspd_mean')
        daily_agg_weather.columns = ['_'.join(col).strip('_') for col in daily_agg_weather.columns.values]
        print("Daily aggregated weather data calculated.")

        # Plot Aggregated Weather
        weather_cols_to_plot = daily_agg_weather.columns.tolist()
        n_plots = len(weather_cols_to_plot)
        n_cols = 1 # Plot each on its own row for clarity
        fig_w, axes_w = plt.subplots(n_plots, n_cols, figsize=(15, n_plots * 4), sharex=True)
        if n_plots == 1: # Ensure axes_w is always iterable
             axes_w = [axes_w]

        for i, col in enumerate(weather_cols_to_plot):
            daily_agg_weather[col].plot(ax=axes_w[i])
            axes_w[i].set_title(f'Daily Aggregated Weather: {col}')
            axes_w[i].set_ylabel(col)
            axes_w[i].grid(True)

        plt.xlabel('Time')
        plt.tight_layout()
        plt.show()

        temp_col = 'tavg_mean'
        if temp_col in daily_agg_weather.columns:
            temp_series_filled = daily_agg_weather[temp_col].fillna(method='ffill').fillna(0) # Ffill then 0
            if len(temp_series_filled) > 365 * 2:
                print(f"\nDecomposing aggregated daily temperature ({temp_col})...")
                try:
                    decomposition_temp = seasonal_decompose(temp_series_filled, model='additive', period=365)
                    fig_decomp_temp = decomposition_temp.plot()
                    fig_decomp_temp.set_size_inches(14, 10)
                    plt.suptitle(f'Decomposition: Daily Aggregated {temp_col}', y=1.01)
                    plt.show()
                except Exception as e:
                    print(f"Could not decompose temperature series: {e}")
            else:
                print("\nSkipping temperature decomposition: Not enough data (< 2 years).")

    else:
        print("No valid weather columns found for aggregation and plotting.")
    gc.collect()

In [ ]:
# 2. Autocorrelation (ACF and PACF)
# Use the aggregated series
print("\nPlotting ACF and PACF for Aggregated Daily Mean Outages...")
# Limit lags shown for clarity
lags = min(60, len(daily_mean_outages_filled)//2 - 1) # Ensure lags < N/2

if lags > 0:
    fig, ax = plt.subplots(1, 2, figsize=(16, 5))
    plot_acf(daily_mean_outages_filled, lags=lags, ax=ax[0])
    ax[0].set_title('Autocorrelation Function (ACF)')
    plot_pacf(daily_mean_outages_filled, lags=lags, ax=ax[1])
    ax[1].set_title('Partial Autocorrelation Function (PACF)')
    plt.tight_layout()
    plt.show()
    print("ACF/PACF plots help identify autocorrelation and seasonality patterns.")
else:
    print("Skipping ACF/PACF plot: Not enough data points.")

In [ ]:
# 3. Lag Analysis (Cross-Correlation) - Simplified Approach
# Calculate correlations between lagged weather features and current outages
print("Lag Analysis (Simplified Cross-Correlation)")
print("Calculating correlation between lagged weather and current outages...")

# Focus on key predictors identified earlier or known impact factors
lag_predictors = ['wspd', 'prcp', 'snow', 'tmax']

# Select one or more high-impact event counts if identified
if 'target_event_corr' in locals():
    high_corr_events = target_event_corr['log_customers_out'].abs().nlargest(10).index.tolist()
    lag_predictors.extend(high_corr_events)

# Ensure predictors are unique and exist
lag_predictors = list(set([p for p in lag_predictors if p in df.columns]))

if not lag_predictors:
    print("No key predictors identified or found for lag analysis.")
else:
    max_lag = 7 # Check lags up to 7 days
    lag_corr_results = {}

    # Use log_customers_out as target for potentially better linear relationship detection
    lag_target = 'log_customers_out'

    for predictor in lag_predictors:
        corrs = {}
        for lag in range(max_lag + 1): # Include lag 0 (contemporary)
            lagged_col_name = f'{predictor}_lag{lag}'
            # Shift introduces NaNs at the beginning, which corr() handles by default
            # Group by county before shifting to prevent data leakage across counties?
            # Simple approach first: shift the whole column
            df[lagged_col_name] = df[predictor].shift(lag)

            # Calculate Spearman correlation (robust)
            valid_data = df[[lag_target, lagged_col_name]].dropna()
            if len(valid_data) > 1:
                corr, _ = spearmanr(valid_data[lag_target], valid_data[lagged_col_name])
                corrs[lag] = corr
            else:
                corrs[lag] = np.nan

            # Clean up the temporary lagged column
            # del df[lagged_col_name]

        lag_corr_results[predictor] = corrs

    # Plot the results
    lag_corr_df = pd.DataFrame(lag_corr_results)
    print("\nSpearman Correlation of log(Outages+1) with Lagged Predictors:")
    print(lag_corr_df)

    lag_corr_df.plot(marker='o', figsize=(14, 7))
    plt.title('Lagged Correlation: Weather/Events vs log(Outages+1)')
    plt.xlabel('Lag (Days)')
    plt.ylabel('Spearman Correlation')
    plt.grid(True)
    plt.axhline(0, color='black', linestyle='--', linewidth=0.7)
    plt.legend(title='Predictor')
    plt.show()
    print(f"This plot shows how strongly weather/events from {max_lag} days ago correlate with today's outages.")
    print("Peaks indicate potentially useful lags for feature engineering.")

    # Cleanup lagged columns created for analysis
    lagged_cols_to_drop = [f'{predictor}_lag{lag}' for predictor in lag_predictors for lag in range(max_lag + 1) if f'{predictor}_lag{lag}' in df.columns]
    if lagged_cols_to_drop:
        df.drop(columns=lagged_cols_to_drop, inplace=True)
        print("Cleaned up temporary lagged columns.")

In [ ]:
# 4. Stationarity Check (Example on aggregated series)
print("Stationarity Check (ADF Test)")

print("Performing Augmented Dickey-Fuller test on aggregated daily mean outages...")
try:
    adf_result = adfuller(daily_mean_outages_filled)
    print(f'ADF Statistic: {adf_result[0]}')
    print(f'p-value: {adf_result[1]}')
    print('Critical Values:')
    for key, value in adf_result[4].items():
        print(f'\t{key}: {value}')

    if adf_result[1] <= 0.05:
        print("Result: The time series is likely stationary (Reject H0).")
    else:
        print("Result: The time series is likely non-stationary (Fail to reject H0). Differencing might be needed for some models.")
except Exception as e:
    print(f"ADF test failed: {e}. Check data length and NaNs.")

---
## E. Geospatial Analysis
---

### 2. Geospatial Patterns: Weather by State/County


In [ ]:
print("2. Geospatial Patterns: Weather by State/County")

# Aggregate weather statistics by State
if 'state' in geo_cols:
    print("\nCalculating average and extreme weather statistics per state...")
    state_weather_agg = df.groupby('state')[weather_num_cols].agg(['mean', 'median', 'max', 'min', 'std']).reset_index()
    # Flatten MultiIndex columns
    state_weather_agg.columns = ['_'.join(col).strip('_') for col in state_weather_agg.columns.values]

    print("\nState Weather Summary (Mean Values):")
    print(state_weather_agg[[col for col in state_weather_agg.columns if 'mean' in col or col == 'state']].head())

    # Identify states with highest max wind speed, precipitation, snow
    print("\nTop 5 States by Max Recorded Wind Speed:")
    if 'wspd_max' in state_weather_agg.columns:
      print(state_weather_agg.sort_values('wspd_max', ascending=False).head(5)[['state', 'wspd_max']])
    else: print("'wspd_max' not found.")


    print("\nTop 5 States by Max Recorded Precipitation:")
    if 'prcp_max' in state_weather_agg.columns:
      print(state_weather_agg.sort_values('prcp_max', ascending=False).head(5)[['state', 'prcp_max']])
    else: print("'prcp_max' not found.")

    print("\nTop 5 States by Max Recorded Snow:")
    if 'snow_max' in state_weather_agg.columns:
      print(state_weather_agg.sort_values('snow_max', ascending=False).head(5)[['state', 'snow_max']])
    else: print("'snow_max' not found.")


    # Plotting distribution of a key weather variable across states (e.g., Mean Wind Speed)
    if 'wspd_mean' in state_weather_agg.columns:
        plt.figure(figsize=(15, 7))
        # Plot top/bottom N states for clarity
        top_n = 30
        plot_data = state_weather_agg.sort_values('wspd_mean', ascending=False) #.head(top_n)
        sns.barplot(data=plot_data, x='state', y='wspd_mean', palette='coolwarm')
        plt.title(f'Mean Wind Speed by State (Top {len(plot_data)})')
        plt.xlabel('State')
        plt.ylabel('Mean Wind Speed (wspd)')
        plt.xticks(rotation=90)
        plt.show()
    else:
        print("Cannot plot mean wind speed by state ('wspd_mean' not found).")

else:
    print("'state' column not found for geospatial weather analysis.")


if 'county' in geo_cols and 'state' in geo_cols and 'wspd' in df.columns:
    county_weather_agg = df.groupby(['state', 'county'])['wspd'].agg(['mean', 'max']).reset_index()
    print("\nTop 10 Counties by Max Recorded Wind Speed:")
    print(county_weather_agg.sort_values('max', ascending=False).head(10))

### 3. Geospatial Patterns: Events by State/County

In [ ]:
print("3. Geospatial Patterns: Events by State/County")

if 'state' in geo_cols and event_cols:
    print("\nCalculating event frequencies per state...")
    # Calculate sum of event occurrences and frequency per state
    state_event_agg = df.groupby('state')[event_cols].agg(['sum', lambda x: (x > 0).mean()]).reset_index()
    # Flatten columns (e.g., 'event_count_xyz_sum', 'event_count_xyz_<lambda>')
    state_event_agg.columns = ['_'.join(col).replace('_<lambda>','_freq').strip('_') for col in state_event_agg.columns.values]

    # Identify events with most geographic variation (highest standard deviation in frequency across states)
    event_freq_cols = [col for col in state_event_agg.columns if col.endswith('_freq')]
    if event_freq_cols:
        event_variation = state_event_agg[event_freq_cols].std().sort_values(ascending=False)
        print("\nTop 5 Events with Most Variation in Frequency Across States:")
        print(event_variation.head())

        # Plot frequency of a high-variation event across states
        most_variable_event_freq = event_variation.index[0] # the top one
        plt.figure(figsize=(15, 7))
        plot_data_event = state_event_agg.sort_values(most_variable_event_freq, ascending=False)
        sns.barplot(data=plot_data_event, x='state', y=most_variable_event_freq, palette='viridis')
        event_name = most_variable_event_freq.replace('_freq', '')
        plt.title(f'Frequency of Days with {event_name} by State')
        plt.xlabel('State')
        plt.ylabel('Frequency (Proportion of Days)')
        plt.xticks(rotation=90)
        plt.show()
    else:
        print("No event frequency columns found for variation analysis.")


    # Find which states have the highest sum of a specific event (e.g., a high-impact event)
    example_event = event_cols[min(9, len(event_cols)-1)] # Pick ~10th event col as example
    example_event_sum_col = f'{example_event}_sum'
    if example_event_sum_col in state_event_agg.columns:
        print(f"\nTop 5 States by Total Occurrences of '{example_event}':")
        print(state_event_agg.sort_values(example_event_sum_col, ascending=False).head(5)[['state', example_event_sum_col]])
    else:
        print(f"Sum column for example event '{example_event}' not found.")

else:
    print("'state' column or event count columns not found for geospatial event analysis.")

## 4. Geospatial Patterns: Severity by State/County


In [ ]:
print("4. Geospatial Patterns: Severity by State/County")

if 'state' in geo_cols and severity_col in df.columns and target_col in df.columns:
    print("\nAnalyzing outage severity distribution per state...")

    # Calculate the proportion of each severity level within each state
    severity_dist = df.groupby('state')[severity_col].value_counts(normalize=True).unstack(fill_value=0)
    print("\nSeverity Distribution (Proportion) per State:")
    print(severity_dist.head())

    # Plot distribution for a few states (top 5 by total outages)
    if 'state_agg' in locals():
         top_states = state_agg.sort_values('total_outages', ascending=False)['state'].head(5).tolist()
    else: # Or just pick first 5 states
        top_states = df['state'].unique()[:5]

    if len(top_states)>0:
        severity_dist.loc[top_states].plot(kind='bar', stacked=True, figsize=(15, 7), colormap='tab10')
        plt.title(f'Severity Distribution in Selected States ({", ".join(top_states)})')
        plt.xlabel('State')
        plt.ylabel('Proportion of County-Days')
        plt.xticks(rotation=0)
        plt.legend(title='Severity', bbox_to_anchor=(1.05, 1), loc='upper left')
        plt.tight_layout(rect=[0, 0, 0.85, 1])
        plt.show()
    else:
        print("No states selected for plotting severity distribution.")


    # Calculate average outage magnitude *when outages occur*
    print("\nCalculating mean outage magnitude (on days with outages > 0) per state...")
    df_with_outages = df[df[target_col] > 0].copy()
    if not df_with_outages.empty:
        mean_magnitude_state = df_with_outages.groupby('state')[target_col].mean().sort_values(ascending=False)
        print("\nTop 10 States by Mean Outage Magnitude (when > 0):")
        print(mean_magnitude_state.head(10))

        # Plot this
        plt.figure(figsize=(15, 7))
        plot_data_mag = mean_magnitude_state.reset_index() #.head(top_n)
        sns.barplot(data=plot_data_mag, x='state', y=target_col, palette='coolwarm')
        plt.title('Mean Outage Magnitude (Customers Out on days with >0 outages) by State')
        plt.xlabel('State')
        plt.ylabel(f'Mean {target_col} (when > 0)')
        plt.xticks(rotation=90)
        plt.show()
    else:
        print("No days with outages found to calculate mean magnitude.")

else:
    print("Required columns ('state', 'severity', 'customers_out') not found for severity analysis.")

## Geospatial Analysis (Using GeoPandas)


In [ ]:
print("E. Geospatial Analysis (Using GeoPandas)")

if not GEOPANDAS_AVAILABLE:
    print("Skipping GeoPandas analysis as the library is not available.")
else:

    shapefile_path = '/content/cb_2020_us_county_500k.shp'
    shapefile_fips_col = 'GEOID' # GEOID or FIPS

    # Load Shapefile
    try:
        print(f"Loading shapefile from: {shapefile_path}")
        gdf = gpd.read_file(shapefile_path)
        print(f"Shapefile loaded successfully. Shape: {gdf.shape}")
        # print(f"Shapefile columns: {gdf.columns.tolist()}")

        # Data Aggregation per County (with Temperature)
        print("Aggregating outage and weather data (including temperature) per county...")
        if 'fips_code' not in df.columns:
             raise ValueError("'fips_code' column is required in the main DataFrame for merging.")

        # Add temperature aggregations
        agg_dict = {
            'total_outages': (target_col, 'sum'),
            'mean_outages': (target_col, 'mean'),
            'median_outages': (target_col, 'median'),
            'mean_log_outages': (log_target_col, 'mean'),
            'max_outages': (target_col, 'max'),
            'days_with_outages': (target_col, lambda x: (x > 0).sum()),
            'record_count': (target_col, 'count'),
            'mean_wspd': ('wspd', 'mean'),
            'mean_tavg': ('tavg', 'mean'),
            'median_tavg': ('tavg', 'median'),
            'max_tmax': ('tmax', 'max'), # Overall max temperature recorded
            'min_tmin': ('tmin', 'min')  # Overall min temperature recorded
        }

        # Filter dictionary for columns actually present in df
        valid_agg_dict = {
            new_col: (orig_col, func)
            for new_col, (orig_col, func) in agg_dict.items()
            if orig_col in df.columns
        }
        print(f"Performing aggregation for columns: {[v[0] for v in valid_agg_dict.values()]}")

        if not valid_agg_dict:
            raise ValueError("No valid columns found for aggregation.")

        county_agg = df.groupby('fips_code').agg(**valid_agg_dict).reset_index()

        # Calculate frequency separately as it uses an aggregated column
        if 'days_with_outages' in county_agg.columns and 'record_count' in county_agg.columns:
            county_agg['outage_frequency'] = county_agg['days_with_outages'] / county_agg['record_count']
        else:
             print("Warning: Could not calculate outage_frequency.")

        print(f"County aggregated data shape: {county_agg.shape}")
        # print(county_agg.head())

        # Prepare for Merge
        # Check if the specified FIPS column exists in the shapefile
        if shapefile_fips_col not in gdf.columns:
            print(f"Warning: Specified FIPS column '{shapefile_fips_col}' not found in shapefile.")
            print("Available columns:", gdf.columns.tolist())
            # Attempt common alternatives or creation if possible
            if 'GEOID' in gdf.columns:
                print("Trying 'GEOID' as FIPS column.")
                shapefile_fips_col = 'GEOID'
            elif 'FIPS' in gdf.columns:
                 print("Trying 'FIPS' as FIPS column.")
                 shapefile_fips_col = 'FIPS'
            elif 'STATEFP' in gdf.columns and 'COUNTYFP' in gdf.columns:
                 print("Attempting to create FIPS from STATEFP+COUNTYFP.")
                 gdf[shapefile_fips_col] = gdf['STATEFP'].astype(str).str.zfill(2) + gdf['COUNTYFP'].astype(str).str.zfill(3)
                 print(f"Created '{shapefile_fips_col}' column in GeoDataFrame.")
            else:
                 raise ValueError(f"Cannot find or create a suitable FIPS column in the shapefile.")

        # Ensure FIPS codes are strings for merging
        gdf[shapefile_fips_col] = gdf[shapefile_fips_col].astype(str).str.strip()
        county_agg['fips_code'] = county_agg['fips_code'].astype(str).str.strip()

        excluded_states = ['02', '15', '60', '66', '69', '72', '78']
        state_fips_col = 'STATEFP'
        if state_fips_col in gdf.columns:
            print("Filtering GeoDataFrame to Contiguous US (CONUS)...")
            gdf_conus = gdf[~gdf[state_fips_col].isin(excluded_states)].copy()
            if gdf_conus.empty: gdf_conus = gdf # Fallback
        else:
            print(f"Warning: State FIPS column '{state_fips_col}' not found. Cannot filter to CONUS.")
            gdf_conus = gdf

        # Merge GeoDataFrame with Aggregated Data
        print(f"Merging GeoDataFrame with aggregated data using FIPS code ('{shapefile_fips_col}' and 'fips_code')...")
        merged_gdf = gdf_conus.merge(county_agg, left_on=shapefile_fips_col, right_on='fips_code', how='left')
        print(f"Merged GeoDataFrame shape: {merged_gdf.shape}")

        # Check merge success
        merge_check_col = 'mean_log_outages'
        if merge_check_col in merged_gdf.columns:
            num_merged = merged_gdf[merge_check_col].notna().sum()
            print(f"Successfully merged data for {num_merged} out of {len(gdf_conus)} CONUS counties.")
            if num_merged == 0: print("Merge Warning: No counties matched. Check FIPS codes.")
        else:
             num_merged = 0
             print(f"Warning: Merge check column '{merge_check_col}' not found after merge.")


        # Create Choropleth Maps
        if num_merged > 0:
            print("\nCreating Choropleth Maps...")

            # Map 1: Mean Log Outages
            plot_col = 'mean_log_outages'
            if plot_col in merged_gdf.columns:
                fig1, ax1 = plt.subplots(1, 1, figsize=(15, 10))
                merged_gdf.plot(column=plot_col, ax=ax1, legend=True, cmap='viridis',
                                missing_kwds={'color': 'lightgrey'},
                                legend_kwds={'label': f"Mean {log_target_col} per County-Day", 'orientation': "horizontal"})
                ax1.set_title(f'Mean {log_target_col} by County (2015-2024)')
                ax1.set_axis_off()
                plt.show()
            else: print(f"Skipping map for {plot_col}, column not found.")

            # Map 2: Outage Frequency
            plot_col = 'outage_frequency'
            if plot_col in merged_gdf.columns:
                fig2, ax2 = plt.subplots(1, 1, figsize=(15, 10))
                merged_gdf.plot(column=plot_col, ax=ax2, legend=True, cmap='plasma',
                                missing_kwds={'color': 'lightgrey'},
                                legend_kwds={'label': "Frequency of Days with Outages (>0)", 'orientation': "horizontal"})
                ax2.set_title('Outage Frequency by County (2015-2024)')
                ax2.set_axis_off()
                plt.show()
            else: print(f"Skipping map for {plot_col}, column not found.")

            # Map 3: Mean Wind Speed
            plot_col = 'mean_wspd'
            if plot_col in merged_gdf.columns:
                 fig3, ax3 = plt.subplots(1, 1, figsize=(15, 10))
                 merged_gdf.plot(column=plot_col, ax=ax3, legend=True, cmap='Blues', # Changed cmap
                                 missing_kwds={'color': 'lightgrey'},
                                 legend_kwds={'label': "Mean Wind Speed (wspd)", 'orientation': "horizontal"})
                 ax3.set_title('Mean Wind Speed by County (2015-2024)')
                 ax3.set_axis_off()
                 plt.show()
            else: print(f"Skipping map for {plot_col}, column not found.")

            # Map 4: Mean Average Temperature (tavg)
            plot_col = 'mean_tavg'
            if plot_col in merged_gdf.columns:
                 fig4, ax4 = plt.subplots(1, 1, figsize=(15, 10))
                 merged_gdf.plot(column=plot_col, ax=ax4, legend=True, cmap='coolwarm', # Good cmap for temperature
                                 missing_kwds={'color': 'lightgrey'},
                                 legend_kwds={'label': "Mean Average Temperature (°C or °F?)", 'orientation': "horizontal"})
                 ax4.set_title('Mean Average Temperature (tavg) by County (2015-2024)')
                 ax4.set_axis_off()
                 plt.show()
            else: print(f"Skipping map for {plot_col}, column not found.")

            # Map 5: Max Recorded Temperature (tmax)
            plot_col = 'max_tmax'
            if plot_col in merged_gdf.columns:
                 fig5, ax5 = plt.subplots(1, 1, figsize=(15, 10))
                 merged_gdf.plot(column=plot_col, ax=ax5, legend=True, cmap='hot_r', # Reversed hot cmap for max temp
                                 missing_kwds={'color': 'lightgrey'},
                                 legend_kwds={'label': "Highest Recorded Max Temperature (°C or °F?)", 'orientation': "horizontal"})
                 ax5.set_title('Maximum Recorded Temperature (tmax) by County (2015-2024)')
                 ax5.set_axis_off()
                 plt.show()
            else: print(f"Skipping map for {plot_col}, column not found.")

            # Map 6: Min Recorded Temperature (tmin)
            plot_col = 'min_tmin'
            if plot_col in merged_gdf.columns:
                 fig6, ax6 = plt.subplots(1, 1, figsize=(15, 10))
                 merged_gdf.plot(column=plot_col, ax=ax6, legend=True, cmap='winter', # Winter cmap for min temp
                                 missing_kwds={'color': 'lightgrey'},
                                 legend_kwds={'label': "Lowest Recorded Min Temperature (°C or °F?)", 'orientation': "horizontal"})
                 ax6.set_title('Minimum Recorded Temperature (tmin) by County (2015-2024)')
                 ax6.set_axis_off()
                 plt.show()
            else: print(f"Skipping map for {plot_col}, column not found.")

        else:
            print("Skipping map creation as no data was successfully merged.")

    except FileNotFoundError:
        print(f"Error: Shapefile not found at '{shapefile_path}'.")
        print("Please update the 'shapefile_path' variable.")
    except ValueError as ve: # Catch specific errors like missing columns
        print(f"ValueError during Geospatial analysis: {ve}")
    except Exception as e:
        print(f"An unexpected error occurred during GeoPandas analysis: {e}")
        import traceback
        traceback.print_exc()

    gc.collect() # Clean up memory

## F. Focus on Extreme/Rare Events

In [ ]:
print("Analysis Focused on Extreme/Rare Events")

# 1. Identify Extreme Thresholds
print("Identifying extreme weather thresholds (e.g., 95th percentile)...")
extreme_thresholds = {}
weather_cols_for_extremes = ['wspd', 'prcp', 'snow', 'tmax'] # Focus on potentially damaging weather
for col in weather_cols_for_extremes:
    if col in df.columns:
        # Consider threshold based on non-zero values for sparse data like snow/prcp
        if col in ['prcp', 'snow']:
            non_zero_values = df[df[col] > 0][col]
            if not non_zero_values.empty:
                extreme_thresholds[col] = non_zero_values.quantile(0.95)
            else:
                 extreme_thresholds[col] = 0 # Or handle differently if no non-zero values
        else:
            extreme_thresholds[col] = df[col].quantile(0.95)
        print(f"  - {col} > {extreme_thresholds[col]:.2f} (95th percentile{' of non-zero values' if col in ['prcp','snow'] and not non_zero_values.empty else ''})")


rare_event_cols = []
if 'target_event_corr' in locals(): # Use least frequent but potentially high impact
    rare_event_cols = sparsity_series[sparsity_series < 1.0].head(10).index.tolist() # Example: events occurring < 1% of days
    print(f"\nIdentifying rare event occurrences (e.g., <1% frequency): {rare_event_cols}")

In [ ]:
# 2. Analyze Outages During Extremes
print("\nComparing outage statistics on extreme vs non-extreme days...")

# Create boolean masks for extreme conditions
extreme_wspd_mask = df['wspd'] > extreme_thresholds.get('wspd', np.inf)
extreme_prcp_mask = df['prcp'] > extreme_thresholds.get('prcp', np.inf)
extreme_snow_mask = df['snow'] > extreme_thresholds.get('snow', np.inf)
extreme_tmax_mask = df['tmax'] > extreme_thresholds.get('tmax', np.inf) # Extreme heat

# Example: Combine conditions - e.g., high wind OR heavy precip OR heavy snow
extreme_weather_mask = extreme_wspd_mask | extreme_prcp_mask | extreme_snow_mask # Add others as needed

# Filter for rare event occurrences
rare_event_mask = pd.Series(False, index=df.index)
for event_col in rare_event_cols:
    if event_col in df.columns:
        rare_event_mask = rare_event_mask | (df[event_col] > 0)

# Combine masks: Any extreme weather OR a rare named event
combined_extreme_mask = extreme_weather_mask | rare_event_mask

df_extreme = df[combined_extreme_mask]
df_non_extreme = df[~combined_extreme_mask]

print("\nOutage Statistics (log_customers_out):")
print("--- Non-Extreme Days ---")
if not df_non_extreme.empty:
    print(df_non_extreme['log_customers_out'].describe())
else:
    print("No non-extreme days found.")

print("\n--- Extreme Days (High Weather Quantiles or Rare Events) ---")
if not df_extreme.empty:
    print(df_extreme['log_customers_out'].describe())
else:
    print("No extreme days found based on defined criteria.")


# Visualize comparison
if not df_extreme.empty and not df_non_extreme.empty:
    plt.figure(figsize=(8, 6))
    sns.boxplot(data=[df_non_extreme['log_customers_out'], df_extreme['log_customers_out']],
                palette='pastel')
    plt.xticks([0, 1], ['Non-Extreme Days', 'Extreme Days'])
    plt.ylabel(f'log({target_col}+1)')
    plt.title('Outage Distribution: Extreme vs Non-Extreme Days')
    plt.show()
else:
    print("Skipping extreme vs non-extreme plot due to empty subset(s).")

In [ ]:
# 3. Event Correlation (Revisit specific rare/severe events)
print("\nRevisiting Correlation for specific rare/severe event types (if identified)...")
if 'event_count_Marine Hurricane/Typhoon' in df.columns:
     hurricane_corr, _ = spearmanr(df['log_customers_out'].fillna(0), df['event_count_Marine Hurricane/Typhoon'].fillna(0))
     print(f"Correlation between log(Outages+1) and 'event_count_hurricane': {hurricane_corr:.3f}")
     # Compare mean outages on hurricane days vs non-hurricane days
     mean_outage_hurricane = df.loc[df['event_count_Marine Hurricane/Typhoon'] > 0, 'log_customers_out'].mean()
     mean_outage_no_hurricane = df.loc[df['event_count_Marine Hurricane/Typhoon'] == 0, 'log_customers_out'].mean()
     print(f"  - Mean log(Outages+1) on Hurricane Days: {mean_outage_hurricane:.2f}")
     print(f"  - Mean log(Outages+1) on Non-Hurricane Days: {mean_outage_no_hurricane:.2f}")

---
## G. Unsupervised Learning Approaches (Exploratory)
---
These are more advanced exploratory steps.

In [ ]:
# 1. Outage Anomaly Detection
print("Unsupervised Learning: Outage Anomaly Detection (Exploratory)")
print("Applying Isolation Forest to detect potentially anomalous outage days...")

# Select features for anomaly detection - could be just the target, or target + key predictors
# Using just log_customers_out for simplicity here
features_for_anomaly = ['log_customers_out']
data_for_anomaly = df[features_for_anomaly].dropna()

if not data_for_anomaly.empty:
    # scaler = StandardScaler()
    # data_scaled = scaler.fit_transform(data_for_anomaly)

    # Initialize and fit Isolation Forest
    # contamination='auto' lets the algorithm estimate the proportion of outliers
    iso_forest = IsolationForest(contamination='auto', random_state=42)
    anomaly_preds = iso_forest.fit_predict(data_for_anomaly) # Returns 1 for inliers, -1 for outliers

    # Add anomaly predictions back to the original DataFrame (matching indices)
    df.loc[data_for_anomaly.index, 'is_outage_anomaly'] = anomaly_preds
    df['is_outage_anomaly'] = df['is_outage_anomaly'] == -1 # Convert to boolean True for anomalies

    n_anomalies = df['is_outage_anomaly'].sum()
    print(f"Detected {n_anomalies} potential outage anomalies ({n_anomalies / len(data_for_anomaly) * 100:.2f}% of days with data).")

    # Analyze characteristics of detected anomalies
    if n_anomalies > 0:
        print("\nCharacteristics of Detected Outage Anomalies:")
        anomaly_df = df[df['is_outage_anomaly'] == True]
        print("\nWeather during anomalies:")
        print(anomaly_df[weather_num_cols].describe())
        print("\nEvents during anomalies (sum):")
        if event_cols:
            print(anomaly_df[event_cols].sum().sort_values(ascending=False).head(10))
        else:
            print("No event columns found.")

        # Compare anomalies to non-anomalies
        plt.figure(figsize=(10, 6))
        sns.boxplot(data=df, x='is_outage_anomaly', y='log_customers_out')
        plt.title('log(Outages+1) for Normal vs Detected Anomaly Days')
        plt.xlabel('Is Anomaly?')
        plt.ylabel(f'log({target_col}+1)')
        plt.show()

        # Cross-reference: Do anomalies correspond to days flagged as 'extreme'?
        if 'combined_extreme_mask' in locals():
            print(f"\nOverlap between detected anomalies and defined 'extreme' days:")
            print(pd.crosstab(df['is_outage_anomaly'], combined_extreme_mask))

    else:
        print("No anomalies detected with current settings.")

else:
    print("Skipping anomaly detection: No data available after handling NaNs.")

---
## Addditional EDA
---

### 1. Deeper Dive: Weather -> Target Relationships


In [ ]:
print("\nVisualizing interaction effects (example: wspd vs outages, colored by prcp)...")

# Use a sample for performance, the dataset is large
df_sample = df.sample(min(50000, len(df)), random_state=42) if len(df) > 50000 else df.copy()

if 'wspd' in df.columns and 'prcp' in df.columns:
    plt.figure(figsize=(12, 8))
    scatter = sns.scatterplot(
        data=df_sample,
        x='wspd',
        y=log_target_col,
        hue='prcp',  # Color by precipitation
        size='prcp', # Size by precipitation (optional, can be noisy)
        palette='viridis',
        alpha=0.6,
        edgecolor=None # Cleaner look
    )
    plt.title(f'log({target_col}+1) vs Wind Speed (Colored/Sized by Precipitation)')
    plt.xlabel('Wind Speed (wspd)')
    plt.ylabel(f'log({target_col}+1)')
    # Improve legend if using size
    h, l = scatter.get_legend_handles_labels()
    # Keep only a few representative legend items for hue/size if too many
    plt.legend(h[::len(h)//8+1], l[::len(l)//8+1], title='Precipitation (prcp)', bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout(rect=[0, 0, 0.85, 1]) # Adjust layout for legend
    plt.show()
else:
    print("Skipping interaction plot: 'wspd' or 'prcp' column not found.")

In [ ]:
print("1. Deeper Weather -> Target Analysis ")

# Binning Weather Variables and Analyzing Target Statistics per Bin
print("\nAnalyzing target variable statistics across weather variable bins...")
n_bins = 5 # Number of bins for weather variables

for col in weather_num_cols:
    print(f"\n--- Analyzing by {col} bins ---")
    try:
        # Create bins using quantiles for potentially better distribution
        df[f'{col}_bin'] = pd.qcut(df[col], q=n_bins, labels=False, duplicates='drop')

        # Calculate statistics per bin
        bin_stats = df.groupby(f'{col}_bin').agg(
            mean_outages=(target_col, 'mean'),
            median_outages=(target_col, 'median'),
            mean_log_outages=(log_target_col, 'mean'),
            median_log_outages=(log_target_col, 'median'),
            outage_frequency=(target_col, lambda x: (x > 0).mean()), # Proportion of days with outages
            record_count=(target_col, 'count')
        ).reset_index()

        # Get approximate range for each bin label
        bin_ranges = df.groupby(f'{col}_bin')[col].agg(['min', 'max'])
        bin_stats = bin_stats.merge(bin_ranges, left_on=f'{col}_bin', right_index=True)
        bin_stats['bin_range'] = bin_stats.apply(lambda row: f"({row['min']:.2f} - {row['max']:.2f}]", axis=1)

        print(bin_stats[['bin_range', 'mean_outages', 'median_outages', 'outage_frequency', 'record_count']])

        # Visualize
        plt.figure(figsize=(10, 5))
        sns.barplot(data=bin_stats, x=f'{col}_bin', y='mean_log_outages', palette='viridis')
        plt.title(f'Mean log({target_col}+1) by {col} Quantile Bins')
        plt.xlabel(f'{col} Quantile Bin (0=Lowest)')
        plt.ylabel(f'Mean log({target_col}+1)')
        # Add text labels for ranges
        plt.xticks(ticks=bin_stats.index, labels=[f"{idx}\n{rng}" for idx, rng in zip(bin_stats.index, bin_stats['bin_range'])], rotation=0, ha='center')
        plt.tight_layout()
        plt.show()

        # Visualize distribution within bins (Box Plot)
        plt.figure(figsize=(12, 6))
        sns.boxplot(data=df.dropna(subset=[f'{col}_bin']), x=f'{col}_bin', y=log_target_col, palette='viridis')
        plt.title(f'log({target_col}+1) Distribution by {col} Quantile Bins')
        plt.xlabel(f'{col} Quantile Bin (0=Lowest)')
        plt.ylabel(f'log({target_col}+1)')
        plt.xticks(ticks=bin_stats.index, labels=[f"{idx}\n{rng}" for idx, rng in zip(bin_stats.index, bin_stats['bin_range'])], rotation=0, ha='center')
        plt.tight_layout()
        plt.show()


        # Clean up bin column
        del df[f'{col}_bin']

    except Exception as e:
        print(f"Could not analyze bins for {col}: {e}")
        if f'{col}_bin' in df.columns: del df[f'{col}_bin']

### 5. Deeper Time Series Analysis (Multivariate Focus)


In [ ]:
print("5. Deeper Time Series Analysis")

# Ensure df has a datetime index for easier time series operations
if not isinstance(df.index, pd.DatetimeIndex):
    if 'time' in df.columns:
        print("Setting 'time' column as index for time series analysis.")
        df = df.set_index('time').sort_index()
    else:
        print("Warning: No 'time' column or datetime index found. Time series analysis will be limited.")

# Summing customers_out per day
if isinstance(df.index, pd.DatetimeIndex):
    print("Aggregating target variable daily across all counties...")
    # Ensure log_target_col exists before grouping
    cols_to_agg = [target_col]
    if log_target_col in df.columns:
        cols_to_agg.append(log_target_col)
    else:
         print(f"Warning: '{log_target_col}' not found, using only '{target_col}'.")

    # Handle potential non-numeric types before aggregation
    for col in cols_to_agg:
        if not pd.api.types.is_numeric_dtype(df[col]):
             print(f"Warning: Column '{col}' is not numeric, attempting conversion.")
             df[col] = pd.to_numeric(df[col], errors='coerce')


    daily_agg = df.groupby(df.index)[log_target_col].mean().dropna() # Mean of log is often more stable

    # Re-plot ACF/PACF for the aggregated target variable (focus on log target)
    if not daily_agg.empty:
        print(f"\nPlotting ACF and PACF for aggregated daily '{log_target_col}'...")
        lags = min(60, len(daily_agg)//2 - 1)
        if lags > 1:
            fig, ax = plt.subplots(1, 2, figsize=(16, 5))
            plot_acf(daily_agg, lags=lags, ax=ax[0])
            ax[0].set_title(f'ACF for Aggregated Daily {log_target_col}')
            plot_pacf(daily_agg, lags=lags, ax=ax[1])
            ax[1].set_title(f'PACF for Aggregated Daily {log_target_col}')
            plt.tight_layout()
            plt.show()
            print("These plots confirm the autocorrelation structure of outages themselves.")
        else:
            print("Skipping ACF/PACF: Not enough data points in aggregated series.")

        # **Multivariate Context: Cross-Correlation**
        # Analyze correlation between aggregated target and aggregated predictors
        print("\nAnalyzing Cross-Correlation between aggregated outages and weather...")
        predictors_for_ccf = ['wspd', 'prcp', 'tmax'] # Example predictors
        predictors_for_ccf = [p for p in predictors_for_ccf if p in df.columns]

        if predictors_for_ccf:
            # Aggregate predictors daily (e.g., mean)
            daily_predictors = df.groupby(df.index)[predictors_for_ccf].mean()
            aligned_target, aligned_predictors = daily_agg.align(daily_predictors, join='inner')
            aligned_target = aligned_target.fillna(0) # Or interpolate
            aligned_predictors = aligned_predictors.fillna(method='ffill').fillna(0) # Ffill then 0

            max_lag_ccf = 14 # Lags/leads to check

            if len(aligned_target) > max_lag_ccf * 2:
                for predictor in predictors_for_ccf:
                    print(f"Calculating CCF for {log_target_col} vs {predictor}...")
                    ccf_values = ccf(aligned_target, aligned_predictors[predictor], adjusted=False) # Using statsmodels ccf

                    # Extract relevant lags/leads (-max_lag to +max_lag) centered around 0
                    # statsmodels ccf output: ccf[0] is lag 0, ccf[1] is lag 1 (target vs predictor(t-1))
                    # We need to map this to conventional plots showing leads and lags.
                    lags_range = np.arange(-max_lag_ccf, max_lag_ccf + 1)
                    # Manually compute correlations for negative lags (leads)
                    ccf_symmetric = []
                    for lag in lags_range:
                        # Positive lag k: corr(target(t), predictor(t-k)) -> use ccf[k]
                        # Negative lag -k (lead k): corr(target(t), predictor(t+k)) == corr(target(t-k), predictor(t))
                        # This requires shifting the target *backwards* (or predictor forwards)
                        if lag >= 0:
                            ccf_symmetric.append(ccf_values[lag])
                        else:
                            # Calculate corr(target(t - abs(lag)), predictor(t))
                            #corr_lead, _ = spearmanr(aligned_target.shift(lag).dropna(), aligned_predictors[predictor].dropna())

                            # Align the shifted target with the predictor to ensure they have the same length
                            temp_target_shifted = aligned_target.shift(lag)
                            # Fix: Reindex temp_target_shifted to match aligned_predictors[predictor]'s index
                            temp_target_shifted = temp_target_shifted.reindex(aligned_predictors[predictor].index)

                            # Handle cases with insufficient data after alignment
                            valid_data = pd.concat([temp_target_shifted, aligned_predictors[predictor]], axis=1).dropna()
                            if len(valid_data) < 2:
                                corr_lead = np.nan
                            else:
                                # Calculate Spearman's correlation for the aligned series
                                corr_lead, _ = spearmanr(valid_data.iloc[:, 0], valid_data.iloc[:, 1])
                            ccf_symmetric.append(corr_lead)


                    plt.figure(figsize=(12, 6))
                    plt.stem(lags_range, ccf_symmetric, basefmt=" ") # Stem plot better for CCF
                    plt.axhline(0, color='grey', lw=0.5)
                    # Add confidence bounds (approximate)
                    conf_level = 1.96 / np.sqrt(len(aligned_target))
                    plt.axhline(conf_level, color='red', linestyle='--', lw=0.8)
                    plt.axhline(-conf_level, color='red', linestyle='--', lw=0.8)
                    plt.title(f'Cross-Correlation: {log_target_col} vs {predictor}')
                    plt.xlabel('Lag (Negative = Predictor Leads Target)')
                    plt.ylabel('Correlation')
                    plt.grid(True, axis='x')
                    plt.show()
                    print(f"CCF shows correlation strength at different leads/lags for {predictor}.")
            else:
                print("Skipping CCF plots: Not enough aligned data points.")
        else:
            print("Skipping CCF analysis: No suitable predictor columns found.")

    else:
        print("Skipping time series analysis: Aggregated daily target is empty.")
else:
    print("Skipping time series analysis: DataFrame does not have a DatetimeIndex.")

### 6. Past vs. Future Outages & Lead/Lag Features

In [ ]:
import random

use_sampling = True
sampling_method = 'county'
sample_frac = 0.1
n_counties_sample = 10
random_seed = 42
max_lag_N = 14
max_lead_L = 14
group_col = 'fips_code'

print(f"Analyzing {max_lag_N} past days predicting {max_lead_L} future days.")

if use_sampling:
    print(f"\n--- Applying Sampling (Method: {sampling_method}) ---")
    original_shape = df.shape
    random.seed(random_seed)
    np.random.seed(random_seed)

    if sampling_method == 'fraction':
        print(f"Sampling {sample_frac * 100:.1f}% of rows randomly...")
        df_sampled = df.sample(frac=sample_frac, random_state=random_seed)
        print(f"Sampled DataFrame shape: {df_sampled.shape}")

    elif sampling_method == 'county':
        all_counties = df[group_col].unique()
        if n_counties_sample >= len(all_counties):
            print("Warning: n_counties_sample >= total unique counties. Using all counties.")
            df_sampled = df.copy()
        else:
            print(f"Sampling data for {n_counties_sample} random counties...")
            selected_counties = random.sample(list(all_counties), n_counties_sample)
            df_sampled = df[df[group_col].isin(selected_counties)].copy() # Use .copy() to avoid SettingWithCopyWarning later
            print(f"Selected {len(selected_counties)} counties.")
            print(f"Sampled DataFrame shape: {df_sampled.shape}")
            print(f"Sampled Time Range: {df_sampled.index.min()} to {df_sampled.index.max()}")

    else:
        raise ValueError("Invalid sampling_method. Choose 'fraction' or 'county'.")

    df_analysis  = df_sampled
    del df_sampled
    gc.collect()

else:
    print("\n--- Running analysis on the FULL dataset (no sampling) ---")
    df_analysis  = df.copy()

In [ ]:
print("6. Past vs. Future Outages & Lead/Lag Features ")

# We already examined ACF/PACF for past lags. Let's focus on future lags (leads).

# Create lead features for the target variable (future outages)
max_lead = 14 # days into the future
lead_target_cols = []
print(f"\nCreating lead features for target variables (up to {max_lead} days ahead)...")
target_vars_for_lead = [target_col, log_target_col]
if severity_col in df.columns:
    # Lagging categorical needs care For now, just lag numeric.
    pass

# Need to group by county before shifting to avoid data leakage across counties
# Store original index if it's time
original_index = df_analysis .index if isinstance(df_analysis .index, pd.DatetimeIndex) else None
df_reset = df_analysis .reset_index() # Work with temp df with standard index

# Create lead columns grouped by county
for col in target_vars_for_lead:
    if col in df_reset.columns:
        for k in range(1, max_lead + 1):
            lead_col_name = f'{col}_lead{k}'
            df_reset[lead_col_name] = df_reset.groupby('fips_code')[col].shift(-k) # Negative shift for lead
            lead_target_cols.append(lead_col_name)

# Restore index if it was datetime
if original_index is not None and 'time' in df_reset.columns:
     df_analysis  = df_reset.set_index('time').sort_index()
else:
     df_analysis  = df_reset # Keep modified df if no time index originally

print(f"Created lead columns: {lead_target_cols}")

# Analyze correlation between CURRENT predictors and FUTURE outages
print(f"\nAnalyzing correlation between current predictors and future outages (up to {max_lead} days)...")
predictors_current = weather_num_cols + event_cols + [target_col, log_target_col] # Include current outage value
predictors_current = [p for p in predictors_current if p in df.columns]

if lead_target_cols and predictors_current:
    # Select relevant columns and drop NaNs for correlation
    cols_for_lead_corr = predictors_current + lead_target_cols
    corr_df = df_analysis [cols_for_lead_corr].dropna(subset=lead_target_cols) # Drop rows where future is unknown

    if not corr_df.empty:
        lead_correlation_matrix = corr_df.corr(method='spearman')

        # Focus on correlation between current predictors and future log_target
        future_log_target_cols = [c for c in lead_target_cols if log_target_col in c]
        predictor_vs_future_corr = lead_correlation_matrix.loc[predictors_current, future_log_target_cols]

        plt.figure(figsize=(15, 10))
        sns.heatmap(predictor_vs_future_corr, annot=True, cmap='coolwarm', fmt=".2f", linewidths=.5, cbar=True)
        plt.title(f'Spearman Correlation: Current Predictors vs Future log({target_col}+1)')
        plt.xlabel('Future Outage Lead Time (Days)')
        plt.ylabel('Current Predictor Variable')
        plt.yticks(rotation=0)
        plt.xticks(rotation=45)
        plt.show()

        print("This heatmap shows how well current conditions predict outages 1 to 7 days ahead.")
        # Highlight predictors strongly correlated with future outages
        abs_corr_sum = predictor_vs_future_corr.abs().sum(axis=1).sort_values(ascending=False)
        print("\nTop 5 Current Predictors Correlated with Future Outages (across leads 1-7):")
        print(abs_corr_sum.head())

    else:
        print("Skipping lead correlation: No data available after handling NaNs from shifts.")
else:
    print("Skipping lead correlation: Lead target columns or predictor columns not available.")

# Inference

## Note
If you Look into the full repo you will see that we planed on exploring and creating a semi-foundational model.
we were doing so but due to some unexpected data, computation issues and numerical instabilties caused by data and the models complex computation we had to forget this, it was a promissing approach as it starts with at least 35% accuracy in next 32 days severity prediction but NaN values cause the models to not learn!




Here we are using zero-shot a foundational model.

**We put a lot of compute power but due to the data nature deep learning models are not able to capture it**

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F # For padding if needed
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
import os
import numpy as np
import matplotlib.pyplot as plt
from accelerate import Accelerator
from accelerate.utils import set_seed, gather_object
from collections import defaultdict
import pandas as pd
from typing import Dict, List, Optional

In [ ]:
class StormTimeSeriesDatasetCompatible(Dataset):
    """
    PyTorch Dataset for UNIvARIATE time series forecasting across multiple counties,
    compatible with the TimesFM TimeSeriesDataset format.

    Generates sequences using a sliding window over the target variable for
    each county independently.
    """
    def __init__(self,
                 dataframe: pd.DataFrame,
                 context_window: int,
                 horizon: int,
                 target_col: str,
                 fips_col: str = 'fips_code',
                 time_col: str = 'time',
                 frequency: int = 0):
        """
        Args:
            dataframe (pd.DataFrame): Sorted DataFrame with time series data for all counties.
                                      MUST be sorted by fips_col and then time_col.
            context_window (int): Length of the input sequence (history).
            horizon (int): Length of the output sequence to predict (future).
            target_col (str): Name of the column to use as the univariate time series.
            fips_col (str): Column name identifying the county/group.
            time_col (str): Column name for the timestamp (used for sorting verification).
            frequency (int): Frequency type (0, 1, or 2). Default is 0.
        """
        super().__init__()
        if len(dataframe) == 0:
            raise ValueError("Input DataFrame is empty.")
        if frequency not in [0, 1, 2]:
            raise ValueError("Invalid frequency value. Must be 0, 1, or 2.")
        if target_col not in dataframe.columns:
             raise ValueError(f"Target column '{target_col}' not found in DataFrame.")
        if fips_col not in dataframe.columns:
            raise ValueError(f"FIPS column '{fips_col}' not found in DataFrame.")
        # Optional: Add check if dataframe is sorted?

        self.df = dataframe # Keep reference if needed for debugging, but could be removed
        self.context_window = context_window
        self.horizon = horizon
        self.target_col = target_col
        self.fips_col = fips_col
        self.time_col = time_col # Keep for index creation logic
        self.frequency = frequency

        # --- Store only the target series as NumPy for efficiency ---
        self.series_data = self.df[self.target_col].astype(np.float32).values

        # --- Create Indices for Sequences (Respecting County Boundaries) ---
        self.indices = self._create_sequence_indices()

        if not self.indices:
             print(f"Warning: No valid sequences found for the given context ({context_window}) "
                   f"and horizon ({horizon}). Check data length per county.")
        else:
             print(f"Created dataset with {len(self.indices)} samples.")

    def _create_sequence_indices(self) -> List[int]:
        """
        Generates a list of valid start indices for sequences across all counties.
        Ensures sequences do not cross county boundaries.
        """
        indices = []
        total_len = len(self.df)
        seq_len = self.context_window + self.horizon

        # Group by county to get start/end row index for each county
        # Using .grouper.groups for potentially better performance if available/needed
        try:
             county_groups = self.df.groupby(self.fips_col, sort=False).indices
        except Exception: # Fallback for older pandas or other issues
             print("Falling back to slower groupby method for indices.")
             county_groups = self.df.groupby(self.fips_col).indices


        print(f"Processing {len(county_groups)} counties...")
        num_skipped = 0
        #skipped_fips = []
        with tqdm(total=len(county_groups), desc="Creating sequence indices") as pbar:
            # Iterate through the unique fips codes in the order they appear
            # Ensure we process groups based on the sorted dataframe structure
            last_fips = None
            group_start_row = 0
            for row_idx, fips in enumerate(self.df[self.fips_col]):
                 if fips != last_fips and last_fips is not None:
                     # Process the completed group
                     county_len = row_idx - group_start_row
                     if county_len >= seq_len:
                         for i in range(county_len - seq_len + 1):
                             abs_start_idx = group_start_row + i
                             indices.append(abs_start_idx)
                     else:
                         num_skipped += 1
                         #skipped_fips.append(last_fips)
                     # Start new group
                     group_start_row = row_idx
                     pbar.update(1)

                 last_fips = fips

            # Process the very last group
            county_len = len(self.df) - group_start_row
            if county_len >= seq_len:
                for i in range(county_len - seq_len + 1):
                    abs_start_idx = group_start_row + i
                    indices.append(abs_start_idx)
            else:
                 num_skipped += 1
                 #skipped_fips.append(last_fips)
            pbar.update(1) # Update for the last group

        print("Finished creating indices.")
        if num_skipped > 0:
            print(f"Skipped {num_skipped} counties/groups that were too short "
                  f"(length < {seq_len}).")
            #print(f"Skipped fips (first few): {skipped_fips[:10]}...")
        return indices

    def __len__(self) -> int:
        """Returns the total number of valid sequences across all counties."""
        return len(self.indices)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
        """
        Retrieves a single sample conforming to the TimesFM TimeSeriesDataset format.

        Args:
            idx (int): Index of the sample to retrieve.

        Returns:
            Tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
                - x_context: Input time series (context_window,)
                - input_padding: Zero padding tensor (context_window,)
                - freq: Frequency tensor ([frequency_value],) shape (1,)
                - x_future: Target time series (horizon,)
        """
        # 1. Get the start index of the sequence in the original DataFrame's series_data
        start_idx = self.indices[idx]

        # 2. Define slice boundaries
        input_end_idx = start_idx + self.context_window
        target_end_idx = input_end_idx + self.horizon

        # 3. Slice the pre-converted NumPy univariate series data
        x_context_np = self.series_data[start_idx:input_end_idx]
        x_future_np = self.series_data[input_end_idx:target_end_idx]

        # --- Verification (Optional but recommended during debugging) ---
        # fips_in_sequence = self.df.iloc[start_idx:target_end_idx][self.fips_col].nunique()
        # if fips_in_sequence > 1:
        #     print(f"Warning: Sequence at index {idx} (starts at df row {start_idx}) "
        #             f"spans multiple fips codes ({self.df.iloc[start_idx:target_end_idx][self.fips_col].unique()})! "
        #             f"Check sorting and _create_sequence_indices.")
        # ---

        # 4. Convert slices to Tensors
        x_context = torch.tensor(x_context_np, dtype=torch.float32)
        x_future = torch.tensor(x_future_np, dtype=torch.float32)

        # 5. Create zero padding with the same shape as the context window
        input_padding = torch.zeros_like(x_context)

        # 6. Create frequency tensor
        freq = torch.tensor([self.frequency], dtype=torch.long)

        # 7. Return the required tuple
        return x_context, input_padding, freq, x_future

In [ ]:
df_test = df_test.fillna(0)

df_test['time'] = pd.to_datetime(df_test['time'])

df_test = df_test.sort_values(by=['fips_code', 'time']).reset_index(drop=True)

target_col = 'customers_out'


In [ ]:
!pip install -q timesfm[torch]

In [ ]:
import timesfm
from timesfm import TimesFm, TimesFmCheckpoint, TimesFmHparams
from timesfm.pytorch_patched_decoder import PatchedTimeSeriesDecoder

In [ ]:
def prepare_model(config: TimesFmConfig, load_weights: bool = False, repo_id: str = "google/timesfm-2.0-500m-pytorch",):
  device = "cuda" if torch.cuda.is_available() else "cpu"
  hparams = TimesFmHparams(
    backend=device,
    per_core_batch_size=32,
    horizon_len=HORIZON,
    num_layers=50,
    use_positional_embedding=False,
    context_len=CONTEXT_WINDOW,
  )
  tfm = TimesFm(hparams=hparams, checkpoint=TimesFmCheckpoint(huggingface_repo_id=repo_id))

  model = PatchedTimeSeriesDecoder(tfm._model_config)
  if load_weights:
    checkpoint_path = os.path.join(snapshot_download(repo_id), "torch_model.ckpt")
    loaded_checkpoint = torch.load(checkpoint_path, weights_only=True)
    model.load_state_dict(loaded_checkpoint)
  return model, hparams, tfm._model_config

In [ ]:
quantiles_list = (0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9)

try:
  median_quantile_index = quantiles_list.index(0.5) + 1
  print(f"Using median quantile at index: {median_quantile_index}")
except ValueError:
  raise ValueError("Median quantile (0.5) not found in quantiles list.")

In [ ]:
def plot_prediction_samples(
    predictions: List[np.ndarray],
    targets: List[np.ndarray],
    context: Optional[List[np.ndarray]] = None,
    samples_to_plot: int = 5,
    save_dir: Optional[str] = None,
    prefix: str = "prediction_sample",
    quantiles_low: Optional[List[np.ndarray]] = None,
    quantiles_high: Optional[List[np.ndarray]] = None,
    context_len : Optional[int] = None,
):
    """
    Plots samples of predictions against targets.

    Args:
        predictions: List of prediction arrays (horizon length).
        targets: List of target arrays (horizon length).
        context: Optional list of context arrays.
        samples_to_plot: Max number of samples to plot.
        save_dir: Directory to save plots. If None, plots are shown.
        prefix: Prefix for saved plot filenames.
        quantiles_low: Optional list of lower quantile prediction arrays.
        quantiles_high: Optional list of upper quantile prediction arrays.
        context_len: Length of the context window for x-axis calculation.
    """
    num_samples = len(predictions)
    samples_to_plot = min(samples_to_plot, num_samples)
    if samples_to_plot == 0:
        print("No samples available to plot.")
        return

    print(f"Plotting {samples_to_plot} prediction samples...")

    # Determine context length from data if not provided
    if context_len is None and context:
         context_len = len(context[0]) if context else 0

    horizon_len = len(predictions[0])
    time_steps_future = np.arange(horizon_len)
    if context_len:
         time_steps_future += context_len # Shift future steps if context is plotted
         time_steps_context = np.arange(context_len)


    for i in range(samples_to_plot):
        plt.figure(figsize=(12, 6))
        plt.plot(time_steps_future, targets[i], label='Actual Target', color='black', marker='.')
        plt.plot(time_steps_future, predictions[i], label='Prediction (Mean/Median)', color='blue', linestyle='--', marker='x')

        # Plot context if available
        if context and context_len and i < len(context):
            plt.plot(time_steps_context, context[i], label='Context', color='gray', linestyle=':')

        # Plot prediction intervals if available
        plot_quantiles = quantiles_low and quantiles_high and i < len(quantiles_low) and i < len(quantiles_high)
        if plot_quantiles:
             plt.fill_between(
                 time_steps_future,
                 quantiles_low[i],
                 quantiles_high[i],
                 color='blue',
                 alpha=0.2,
                 label=f'Quantile Interval' # Add specific quantile info if known
             )

        plt.title(f'Prediction vs Actual - Sample {i+1}')
        plt.xlabel('Time Step')
        plt.ylabel('Value')
        plt.legend()
        plt.grid(True, linestyle='--', alpha=0.6)

        if save_dir:
            os.makedirs(save_dir, exist_ok=True)
            save_path = os.path.join(save_dir, f"{prefix}_{i+1}.png")
            plt.savefig(save_path, bbox_inches='tight')
            plt.close() # Close figure after saving
        else:
            plt.show()

In [ ]:
import torch
from torch import nn
import torch.nn.functional as F
from typing import List, Tuple, Dict, Optional

import os
# import pandas as pd # Not strictly needed for these functions
import numpy as np
import matplotlib.pyplot as plt

# --- Constants based on your example ---
# You might want to define these elsewhere or pass them if they vary
PREDICTION_HORIZON = 32

def calculate_standard_metrics(predictions: torch.Tensor, target: torch.Tensor, epsilon=1e-10) -> Dict[str, torch.Tensor]:
    """
    Calculates standard time series metrics (MAE, MSE, RMSE, MAPE).

    Args:
        predictions (torch.Tensor): The point forecast tensor.
                                     Expected shape: [B, H] (e.g., your last_patch_pred).
        target (torch.Tensor): The ground truth tensor.
                               Expected shape: [B, H] or [B, H, 1] (e.g., your x_future).
        epsilon (float): Small value to avoid division by zero in MAPE.

    Returns:
        Dict[str, torch.Tensor]: Dictionary containing scalar metric tensors ('mae', 'mse', 'rmse', 'mape').
    """
    # Ensure predictions are point forecasts [B, H]
    if predictions.ndim != 2:
         raise ValueError(f"Predictions tensor should have shape [B, H], but got {predictions.shape}")

    # Ensure target is [B, H]
    if target.shape == predictions.shape:
        y_true = target
    elif target.ndim == 3 and target.shape[-1] == 1 and target.shape[:2] == predictions.shape:
        y_true = target.squeeze(-1)
    else:
         raise ValueError(f"Target shape mismatch or unexpected shape. Pred: {predictions.shape}, Target: {target.shape}. Target should be [B, H] or [B, H, 1].")

    y_pred_point = predictions.detach().float() # Ensure float and detach
    y_true = y_true.detach().float() # Ensure float and detach

    if y_pred_point.shape != y_true.shape:
         # This check should ideally be caught by the initial checks, but kept for safety
         raise ValueError(f"Internal shape mismatch after processing: Pred {y_pred_point.shape}, Target {y_true.shape}")

    abs_error = torch.abs(y_true - y_pred_point)
    sq_error = abs_error ** 2

    # Calculate metrics as tensors (NO .item()) - mean over batch and horizon
    mae = torch.mean(abs_error)
    mse = torch.mean(sq_error)
    rmse = torch.sqrt(mse)
    mape = torch.mean(torch.div(abs_error, torch.abs(y_true) + epsilon)) * 100

    # Return dictionary of scalar tensors
    return {'mae': mae, 'mse': mse, 'rmse': rmse, 'mape': mape}


def map_to_severity(values: torch.Tensor, boundaries: torch.Tensor) -> torch.Tensor:
    """Maps continuous values to discrete severity levels based on boundaries."""
    # Ensure boundaries are on the same device as values
    boundaries = boundaries.to(values.device)
    # Find the bucket index for each value. Levels are 0, 1, 2,...
    severity_levels = torch.bucketize(values, boundaries, right=False)
    return severity_levels.long() # Return as long integers

def calculate_severity_accuracy(predictions: torch.Tensor,
                                target: torch.Tensor,
                                boundaries: torch.Tensor) -> dict:
    """
    Calculates severity accuracy based on predefined boundaries.

    Args:
        predictions (torch.Tensor): The point forecast tensor used for severity classification
                                     (e.g., median or mean prediction).
                                     Expected shape: [B, H] (e.g., your last_patch_pred).
        target (torch.Tensor): The ground truth tensor.
                               Expected shape: [B, H] or [B, H, 1] (e.g., your x_future).
        boundaries (torch.Tensor): Tensor defining the thresholds for severity levels.
                                    E.g., torch.tensor([10, 20, 30]) defines levels:
                                    (-inf, 10), [10, 20), [20, 30), [30, inf) -> levels 0, 1, 2, 3

    Returns:
        dict: Dictionary containing:
              'overall_severity_accuracy' (scalar tensor): Accuracy averaged over batch and horizon.
              'severity_accuracy_per_step' (tensor): Accuracy for each step in the horizon. Shape: (H,)
    """
    # Ensure predictions are point forecasts [B, H]
    if predictions.ndim != 2:
         raise ValueError(f"Predictions tensor should have shape [B, H], but got {predictions.shape}")

    # Ensure target is [B, H]
    if target.shape == predictions.shape:
        y_true = target
    elif target.ndim == 3 and target.shape[-1] == 1 and target.shape[:2] == predictions.shape:
        y_true = target.squeeze(-1)
    else:
         raise ValueError(f"Target shape mismatch or unexpected shape. Pred: {predictions.shape}, Target: {target.shape}. Target should be [B, H] or [B, H, 1].")

    y_pred_point_for_severity = predictions.detach().float()
    y_true = y_true.detach().float()

    if y_pred_point_for_severity.shape != y_true.shape:
         raise ValueError(f"Internal shape mismatch after processing: Pred {y_pred_point_for_severity.shape}, Target {y_true.shape}")

    # Map predictions and target to severity levels
    pred_severity = map_to_severity(y_pred_point_for_severity, boundaries)
    actual_severity = map_to_severity(y_true, boundaries)

    # Calculate correctness (element-wise)
    correct = (pred_severity == actual_severity).float() # Convert boolean to float (0.0 or 1.0)

    # Calculate overall accuracy (mean over batch and horizon)
    overall_accuracy = torch.mean(correct) * 100

    # Calculate accuracy per step (mean over batch dim B)
    # correct has shape [B, H], mean over dim 0 gives shape [H]
    accuracy_per_step = torch.mean(correct, dim=0) * 100

    return {
        'overall_severity_accuracy': overall_accuracy,      # scalar tensor
        'severity_accuracy_per_step': accuracy_per_step   # tensor shape (H,)
    }

def plot_severity_accuracy(accuracy_per_step: torch.Tensor, save_path: str = None):
    """
    Plots the severity accuracy over the forecast horizon.

    Args:
        accuracy_per_step (torch.Tensor): Accuracy for each step. Shape: (H,)
        save_path (str, optional): Path to save the plot image. If None, shows plot.
    """
    horizon = len(accuracy_per_step)
    steps = np.arange(horizon) # Steps 0, 1, ..., H-1
    accuracy_values = accuracy_per_step.cpu().numpy() # Move to CPU and convert to NumPy

    plt.figure(figsize=(10, 5))
    plt.plot(steps, accuracy_values, marker='o', linestyle='-')
    plt.title('Severity Level Prediction Accuracy over Forecast Horizon')
    plt.xlabel('Forecast Horizon Step')
    plt.ylabel('Accuracy (%)')
    plt.xticks(steps[::max(1, horizon//10)]) # Adjust tick frequency
    plt.ylim(0, 105)
    plt.grid(True, linestyle='--', alpha=0.7)

    if save_path:
        plt.savefig(save_path, bbox_inches='tight')
        print(f"Severity accuracy plot saved to {save_path}")
        plt.close() # Close the figure after saving
    else:
        plt.show() # Display the plot interactively


def _quantile_loss(pred: torch.Tensor, actual: torch.Tensor, quantile: float) -> torch.Tensor:
    """Calculates quantile loss element-wise.
    Args:
        pred: Predicted values (Batch, Horizon)
        actual: Actual values (Batch, Horizon)
        quantile: Quantile at which loss is computed
    Returns:
        Quantile loss for each element (Batch, Horizon)
    """
    dev = actual - pred
    loss_positive = dev * quantile
    loss_negative = -dev * (1.0 - quantile)
    # Use torch.where for element-wise conditional selection
    element_wise_loss = 2 * torch.where(dev >= 0, loss_positive, loss_negative)
    return element_wise_loss

In [ ]:
def inference_function_timesfm(
    config: Dict,
    results_dir: str = "./inference_results"
):
    """
    Runs inference using a trained TimesFM model on test data with Accelerate.

    Args:
        config (Dict): Dictionary containing configuration (must match training config closely,
                       especially context_window, horizon, frequency, target_col, quantiles if used).
                       Requires 'severity_boundaries' as torch.Tensor.
        results_dir (str): Directory to save metrics and plots.
    """
    set_seed(config['seed'])
    os.makedirs(results_dir, exist_ok=True)

    accelerator = Accelerator(
        # Mixed precision can be used for inference speedup too
        mixed_precision=config['mixed_precision_type'] if config['use_amp'] else "no",
        # No need for log_with during pure inference unless logging metrics to TensorBoard
        # log_with="tensorboard",
        # project_dir=os.path.join(results_dir, "logs") #
    )

    # --- Model Loading ---
    if accelerator.is_main_process:
        print(f"--- TimesFM Inference ---")
        print(f"Loading model structure...")
    model, hparams, _model_config = prepare_model(load_weights = True, repo_id= "google/timesfm-1.0-200m-pytorch" )
    print("Model loaded")

    # --- Dataset and DataLoader ---
    if accelerator.is_main_process:
        print("Preparing test dataset and dataloader...")
    test_dataset = StormTimeSeriesDatasetCompatible(
        dataframe=df_test,
        context_window=config['context_window'],
        horizon=config['horizon'],
        target_col=config['target_col'],
        fips_col='fips_code',
        time_col='time',
        frequency=config['frequency']
    )
    test_dataloader = DataLoader(
        test_dataset,
        shuffle=False, # NEVER shuffle test data
        batch_size=config['batch_size'], # Can potentially increase batch size for inference
        num_workers=config['num_workers'],
        pin_memory=True,
        prefetch_factor=config.get('prefetch_factor', 2)
    )

    # --- Prepare Model and Dataloader with Accelerator ---
    model, test_dataloader = accelerator.prepare(model, test_dataloader)

    # --- Move Severity Boundaries ---
    # Ensure boundaries tensor exists in config and move it to device
    if 'severity_boundaries' not in config or not isinstance(config['severity_boundaries'], torch.Tensor):
        raise ValueError("Config must include 'severity_boundaries' as a torch.Tensor for severity metrics.")
    severity_boundaries_tensor = config['severity_boundaries'].to(accelerator.device)

    # --- Set Model to Evaluation Mode ---
    model.eval()
    if accelerator.is_main_process:
        print(f"Model set to evaluation mode. Running inference on {accelerator.device}...")

    # --- Inference Loop ---
    inference_metrics_batches = defaultdict(list)
    # Store some raw predictions/targets/context from each process for plotting
    collected_samples = defaultdict(list)
    num_samples_to_collect_per_process = 5 # Collect a few samples for visualization

    progress_bar = tqdm(test_dataloader,
                          disable=not accelerator.is_local_main_process,
                          desc="Inference Progress")

    with torch.no_grad():
        for step, batch in enumerate(progress_bar):
            # Data is already on the correct device
            x_context, x_padding, freq, x_future = batch
            target = x_future # Shape [B, H]

            # --- Forward Pass ---
            predictions = model(x_context, x_padding.float(), freq) # [B, NumP, PatchL, OutputD]

            # --- Extract Mean Prediction for Standard Metrics ---
            predictions_mean = predictions[..., 0] # [B, NumP, PatchL]
            last_patch_pred_mean = torch.zeros_like(target) # Fallback
            if predictions_mean.shape[1] > 0:
                patch_len = predictions_mean.shape[2]
                horizon = config['horizon']
                last_patch_pred_mean = predictions_mean[:, -1, :horizon] # [B, H]
                # Pad if patch length < horizon (should match config ideally)
                if last_patch_pred_mean.shape[1] < horizon:
                     pad_width = horizon - last_patch_pred_mean.shape[1]
                     last_patch_pred_mean = F.pad(last_patch_pred_mean, (0, pad_width), "constant", 0)


            # --- Handle Potential Shape Mismatch (Safety Check) ---
            if last_patch_pred_mean.shape != target.shape:
                 if accelerator.is_main_process and step == 0:
                     print(f"Warning: Inference loop - Final shape mismatch. Pred Mean: {last_patch_pred_mean.shape}, Target: {target.shape}. Using zero pred for metrics.")
                 last_patch_pred_mean = torch.zeros_like(target)

            target_float = target.float() # Target for metrics
            pred_for_std_metrics = last_patch_pred_mean.float() # Prediction for std metrics

            # --- Calculate Standard Metrics for this Batch ---
            batch_std_metrics_tensors = calculate_standard_metrics(
                pred_for_std_metrics, target_float
            )
            # Gather each scalar metric tensor across processes
            gathered_mae = accelerator.gather_for_metrics(batch_std_metrics_tensors['mae'])
            gathered_mse = accelerator.gather_for_metrics(batch_std_metrics_tensors['mse'])
            gathered_rmse = accelerator.gather_for_metrics(batch_std_metrics_tensors['rmse'])
            gathered_mape = accelerator.gather_for_metrics(batch_std_metrics_tensors['mape'])
            # Store the *mean* metric across processes for *this batch*
            inference_metrics_batches['mae'].append(gathered_mae.mean().item())
            inference_metrics_batches['mse'].append(gathered_mse.mean().item())
            inference_metrics_batches['rmse'].append(gathered_rmse.mean().item())
            inference_metrics_batches['mape'].append(gathered_mape.mean().item())

            # --- Calculate Severity Accuracy Metrics ---
            pred_for_sev_metrics = None
            quantiles_low_pred = None
            quantiles_high_pred = None
            # Determine prediction for severity and potentially quantiles for plotting
            if config.get('use_quantile_loss', False) and 'median_quantile_index' in config:
                 median_idx_config = config['median_quantile_index']
                 quantile_channel_idx = median_idx_config + 1 # +1 because channel 0 is mean
                 num_outputs = predictions.shape[-1]

                 if predictions.shape[1] > 0 and num_outputs > quantile_channel_idx:
                       median_pred = predictions[:, -1, :config['horizon'], quantile_channel_idx].float()
                       if median_pred.shape[1] < config['horizon']:
                           pad_width = config['horizon'] - median_pred.shape[1]
                           median_pred = F.pad(median_pred, (0, pad_width), "constant", 0)
                       pred_for_sev_metrics = median_pred # Use median for severity

                       # Try to get lower/upper quantiles for plotting if available
                       # Assumes quantiles are ordered [low, median, high] typically
                       if median_idx_config > 0 and (median_idx_config + 2) < num_outputs : # Check bounds
                             quantiles_low_pred = predictions[:, -1, :config['horizon'], median_idx_config].float() # Index before median
                             quantiles_high_pred = predictions[:, -1, :config['horizon'], median_idx_config + 2].float() # Index after median
                             # Padding for quantiles if needed
                             if quantiles_low_pred.shape[1] < config['horizon']:
                                  pad_width = config['horizon'] - quantiles_low_pred.shape[1]
                                  quantiles_low_pred = F.pad(quantiles_low_pred, (0, pad_width), "constant", 0)
                             if quantiles_high_pred.shape[1] < config['horizon']:
                                 pad_width = config['horizon'] - quantiles_high_pred.shape[1]
                                 quantiles_high_pred = F.pad(quantiles_high_pred, (0, pad_width), "constant", 0)

                 else:
                       pred_for_sev_metrics = pred_for_std_metrics # Fallback to mean
            else:
                 pred_for_sev_metrics = pred_for_std_metrics # Use mean if no quantiles

            # --- Calculate Severity Metrics ---
             # Ensure prediction used for severity matches target shape
            if pred_for_sev_metrics.shape != target_float.shape:
                 if accelerator.is_main_process and step == 0:
                     print(f"Warning: Inference loop - Final shape mismatch for severity calculation. Skipping severity for this batch.")
                 batch_sev_metrics_tensors = {'overall_severity_accuracy': torch.tensor(torch.nan, device=accelerator.device),
                                              'severity_accuracy_per_step': torch.full((config['horizon'],), torch.nan, device=accelerator.device)}
            else:
                 batch_sev_metrics_tensors = calculate_severity_accuracy(
                     pred_for_sev_metrics,
                     target_float,
                     severity_boundaries_tensor
                 )

            # Gather Severity Metrics
            sev_acc_overall_tensor = batch_sev_metrics_tensors['overall_severity_accuracy']
            gathered_sev_acc_overall = accelerator.gather_for_metrics(sev_acc_overall_tensor)
            inference_metrics_batches['overall_severity_accuracy'].append(torch.nanmean(gathered_sev_acc_overall).item())

            sev_acc_per_step_tensor = batch_sev_metrics_tensors['severity_accuracy_per_step']
            gathered_sev_acc_per_step = accelerator.gather_for_metrics(sev_acc_per_step_tensor)
            avg_batch_sev_acc_per_step = torch.nanmean(gathered_sev_acc_per_step, dim=0)
            inference_metrics_batches['severity_accuracy_per_step_tensors'].append(avg_batch_sev_acc_per_step)

            # --- Collect Samples for Plotting (from each process) ---
            # Store tensors on CPU to avoid GPU memory buildup if test set is large
            if len(collected_samples['predictions']) < num_samples_to_collect_per_process:
                 collected_samples['predictions'].extend(list(pred_for_sev_metrics.detach().cpu().numpy())) # Use pred used for severity (median or mean)
                 collected_samples['targets'].extend(list(target_float.detach().cpu().numpy()))
                 collected_samples['context'].extend(list(x_context.detach().cpu().numpy()))
                 if quantiles_low_pred is not None and quantiles_high_pred is not None:
                      collected_samples['quantiles_low'].extend(list(quantiles_low_pred.detach().cpu().numpy()))
                      collected_samples['quantiles_high'].extend(list(quantiles_high_pred.detach().cpu().numpy()))


    # --- End of Inference Loop ---

    # --- Aggregate Final Metrics ---
    accelerator.wait_for_everyone()
    if accelerator.is_main_process:
        print("\nInference loop finished. Aggregating metrics...")

    # Average standard metrics and overall severity accuracy
    final_metrics = {}
    for name, values in inference_metrics_batches.items():
         if name != 'severity_accuracy_per_step_tensors':
              valid_values = [v for v in values if not np.isnan(v)]
              final_metrics[name] = np.mean(valid_values) if valid_values else np.nan

    # Average severity accuracy per step
    avg_final_severity_acc_per_step = torch.full((config['horizon'],), torch.nan, device="cpu") # Aggregate on CPU
    sev_step_tensors = inference_metrics_batches.get('severity_accuracy_per_step_tensors', [])
    valid_sev_step_tensors = [t.cpu() for t in sev_step_tensors if not torch.all(torch.isnan(t))] # Move to CPU
    if valid_sev_step_tensors:
        try:
             stacked_sev_step_tensors = torch.stack(valid_sev_step_tensors, dim=0)
             avg_final_severity_acc_per_step = torch.nanmean(stacked_sev_step_tensors, dim=0)
        except Exception as stack_err:
             if accelerator.is_main_process: print(f"Error stacking/averaging final per-step severity tensors: {stack_err}")

    # --- Display and Save Metrics ---
    if accelerator.is_main_process:
        print("\n--- Final Inference Metrics ---")
        metric_lines = []
        for name, value in final_metrics.items():
            print(f"  {name.replace('_', ' ').title()}: {value:.4f}")
            metric_lines.append(f"{name}: {value:.6f}\n") # For saving to file

        # Save metrics to a text file
        metrics_save_path = os.path.join(results_dir, "final_inference_metrics.txt")
        try:
            with open(metrics_save_path, "w") as f:
                f.writelines(metric_lines)
                # Add per-step severity if available
                if not torch.all(torch.isnan(avg_final_severity_acc_per_step)):
                     f.write("\nSeverity Accuracy Per Step:\n")
                     for i, acc in enumerate(avg_final_severity_acc_per_step):
                          f.write(f"  Step {i}: {acc.item():.4f}\n")
            print(f"Metrics saved to: {metrics_save_path}")
        except Exception as save_err:
             print(f"Error saving metrics file: {save_err}")

        # --- Plotting and Interpretation ---
        print("\nGenerating plots...")
        # 1. Plot Severity Accuracy Per Step
        if not torch.all(torch.isnan(avg_final_severity_acc_per_step)):
            sev_plot_path = os.path.join(results_dir, "inference_severity_accuracy_plot.png")
            try:
                plot_severity_accuracy(avg_final_severity_acc_per_step.cpu(), save_path=sev_plot_path)
            except Exception as plot_err:
                 print(f"Could not generate severity plot: {plot_err}")
        else:
             print("Skipping severity plot (NaN results).")

        # 2. Plot Prediction Samples
        # Gather collected samples from all processes
        all_collected_samples = gather_object(collected_samples) # List of dicts from each process

        # Consolidate samples on the main process
        final_plot_samples = defaultdict(list)
        for proc_samples in all_collected_samples:
             for key in proc_samples:
                  final_plot_samples[key].extend(proc_samples[key])


        # Check if quantile keys exist before trying to access them
        plot_quantiles_low = final_plot_samples.get('quantiles_low')
        plot_quantiles_high = final_plot_samples.get('quantiles_high')

        if final_plot_samples['predictions']:
             try:
                 plot_prediction_samples(
                     predictions=final_plot_samples['predictions'],
                     targets=final_plot_samples['targets'],
                     context=final_plot_samples.get('context'), # Use .get for optional context
                     samples_to_plot=10, # Plot up to 10 gathered samples
                     save_dir=os.path.join(results_dir, "prediction_plots"),
                     prefix="inference_prediction_sample",
                     quantiles_low=plot_quantiles_low,
                     quantiles_high=plot_quantiles_high,
                     context_len=config['context_window']
                 )
                 print(f"Prediction sample plots saved to: {os.path.join(results_dir, 'prediction_plots')}")
             except Exception as plot_err:
                  print(f"Could not generate prediction sample plots: {plot_err}")
        else:
             print("No prediction samples were collected for plotting.")

    accelerator.end_training() # Clean up accelerator resources
    if accelerator.is_main_process: print("\nInference function finished.")

In [ ]:
CONTEXT_WINDOW = 192
HORIZON = 32
FREQUENCY = 0
BATCH_SIZE = 2048 # works per L4
EPOCHS = 5
SEED = 42
USE_AMP = True # False
MIXED_PRECISION_TYPE = "bf16" # no if nan
USE_TORCH_COMPILE = hasattr(torch, 'compile')


SEVERITY_BOUNDARIES = torch.tensor([1, 5, 10, 100, 1000, 2000, 5000, 10000, 100000], dtype=torch.float32) # Your boundaries
SAVE_SEVERITY_PLOT = True #
SEVERITY_PLOT_FILENAME = "severity_accuracy_plot.png" # Plot filename

In [ ]:
def get_optimal_num_workers():
    """Returns a suggested number of workers based on CPU cores."""
    num_cores = os.cpu_count()
    if num_cores is None:
        return 4 # Default if cpu_count fails

    optimal_workers = min(num_cores, 16)

    return optimal_workers


def count_parameters(model: torch.nn.Module):
    """Counts total and trainable parameters in a model."""
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total_params, trainable_params

In [ ]:
config = {
    "epochs": EPOCHS,
    "learning_rate": LEARNING_RATE,
    "batch_size": BATCH_SIZE,
    "seed": SEED,
    "use_amp": USE_AMP,
    "mixed_precision_type": MIXED_PRECISION_TYPE,
    "use_torch_compile": USE_TORCH_COMPILE,
    "use_fused_optimizer": USE_FUSED_OPTIMIZER,
    "gradient_clip_norm": GRADIENT_CLIP_NORM,
    "num_workers": get_optimal_num_workers(),
    "prefetch_factor": 2,
    "context_window": CONTEXT_WINDOW,
    "horizon": HORIZON,
    "target_col": target_col,
    "frequency": FREQUENCY,
    "quantiles_list": quantiles_list,
    "logging_dir": "training_logs",
    "project_name": "timesfm-quantile-forecast",
    "output_model_file": "final_storm_model.safetensors",
    "severity_boundaries": SEVERITY_BOUNDARIES,
    "median_quantile_index": median_quantile_index,
    "save_severity_plot": SAVE_SEVERITY_PLOT,
    "severity_plot_filename": SEVERITY_PLOT_FILENAME,
    'use_quantile_loss':False
}

In [ ]:
from accelerate import notebook_launcher

#Use number of your GPUs for process
notebook_launcher(inference_function_timesfm, [config], num_processes=4)

# Additional info

**This was gonna be a great submission but time and data were not our best friend**

**But still check out the full repo**

**We pointed that Simple ML models don't work**

**LSTM based models worked but data and it size cause them to not work.**


repo: https://github.com/Mhdaw/NeuralStorm